In [ ]:
# Mount Google Drive and initialize project paths

from google.colab import drive
from pathlib import Path

import json
import hashlib
import platform
import sys
import gc
import joblib

import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime, timezone
from sklearn.metrics import precision_recall_curve

drive.mount('/content/drive')

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

CONFIG_DIR = PROJECT_ROOT / '00_config'
PROCESSED_DIR = PROJECT_ROOT / '01_data' / 'processed'
SPLITS_DIR = PROJECT_ROOT / '01_data' / 'splits'
FEATURES_DIR = PROJECT_ROOT / '01_data' / 'features'
SRC_DIR = PROJECT_ROOT / '03_src'

CLASSICAL_DIR = (
    PROJECT_ROOT / '02_notebooks' / '03_classical_ml'
)

NOTEBOOK_DIR = CLASSICAL_DIR / '06_classical_ml_comparison'

RESULTS_DIR = NOTEBOOK_DIR / 'results'
TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
LOGS_DIR = RESULTS_DIR / 'logs'
RUN_CONFIGS_DIR = RESULTS_DIR / 'configs'

REGISTRY_PATH = CONFIG_DIR / 'experiment_registry.csv'
DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

assert NOTEBOOK_DIR.is_dir()

for folder in [
    TABLES_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    RUN_CONFIGS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)


# Load shared project configuration

with open(CONFIG_DIR / 'project_config.json', 'r') as file:
    config = json.load(file)

LABELS = config['labels']
SEED = config['seed']

FEATURE_BUDGET = (
    config['classical_features']['feature_budget']
)

assert len(LABELS) == 6
assert SEED == 42
assert FEATURE_BUDGET == 50000
assert config['classical_features']['status'] == 'verified'


# Load the shared evaluation module

sys.path.insert(0, str(SRC_DIR))

import evaluation_utils

from evaluation_utils import (
    apply_thresholds,
    evaluate_multilabel,
    optimize_thresholds
)

assert Path(evaluation_utils.__file__).resolve() == (
    SRC_DIR / 'evaluation_utils.py'
).resolve()


def calculate_sha256(path):

    sha256 = hashlib.sha256()

    with open(path, 'rb') as file:

        for chunk in iter(
            lambda: file.read(1024 * 1024), b''
        ):
            sha256.update(chunk)

    return sha256.hexdigest()


evaluation_hash = calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
)

feature_manifest_hash = calculate_sha256(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
)


# Load the completed classical experiment registry

registry_df = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

expected_run_ids = [
    'NB_WORD_BASELINE_S42',
    'LR_WORD_BASELINE_S42',
    'SVM_WORD_BASELINE_S42',
    'DT_WORD_BASELINE_S42',
    'RF_WORD_BASELINE_S42',
    'LR_CHAR_BASELINE_S42',
    'LR_COMBINED_BASELINE_S42',
    'SVM_CHAR_BASELINE_S42',
    'SVM_COMBINED_BASELINE_S42',
    'LR_COMBINED_BALANCED_S42',
    'SVM_COMBINED_BALANCED_S42'
]

assert registry_df['run_id'].is_unique

assert set(registry_df['run_id']) == set(
    expected_run_ids
)

assert len(registry_df) == 11
assert registry_df['status'].eq('Completed').all()

assert registry_df['seed'].astype(int).eq(SEED).all()

assert registry_df[
    'training_samples'
].astype(int).eq(59997).all()

registry_original_hash = calculate_sha256(
    REGISTRY_PATH
)

decision_log_original_hash = calculate_sha256(
    DECISION_LOG_PATH
)


# Resolve historical threshold configuration formats

def resolve_saved_thresholds(
    threshold_config,
    predictions_path,
    run_id
):

    threshold_keys = [
        'optimized_thresholds',
        'thresholds',
        'optimal_thresholds',
        'per_label_thresholds',
        'optimized_per_label_thresholds',
        'label_thresholds',
        'optimized'
    ]

    saved_predictions = pd.read_parquet(
        predictions_path
    )

    assert saved_predictions['id'].is_unique
    assert len(saved_predictions) == 31911

    saved_scores = saved_predictions[[
        f'{label}_score'
        for label in LABELS
    ]].to_numpy(dtype=np.float64)

    saved_binary_predictions = saved_predictions[[
        f'{label}_pred_optimized'
        for label in LABELS
    ]].to_numpy()

    assert saved_scores.shape == (
        len(saved_predictions),
        len(LABELS)
    )

    assert np.isfinite(saved_scores).all()

    assert np.isin(
        saved_binary_predictions,
        [0, 1]
    ).all()

    valid_candidates = []

    for key in threshold_keys:

        if key not in threshold_config:
            continue

        values = threshold_config[key]

        try:

            if isinstance(values, dict):

                if not all(
                    label in values
                    for label in LABELS
                ):
                    continue

                candidate = np.array([
                    values[label]
                    for label in LABELS
                ], dtype=np.float64)

            elif isinstance(values, list):

                candidate = np.asarray(
                    values,
                    dtype=np.float64
                )

            else:
                continue

        except (TypeError, ValueError):
            continue

        if candidate.shape != (len(LABELS),):
            continue

        if not np.isfinite(candidate).all():
            continue

        reconstructed_predictions = apply_thresholds(
            saved_scores,
            candidate
        )

        if np.array_equal(
            reconstructed_predictions,
            saved_binary_predictions
        ):

            valid_candidates.append(
                (key, candidate)
            )

    if not valid_candidates:

        print("Run ID:", run_id)

        print(
            "Available threshold configuration keys:",
            list(threshold_config.keys())
        )

        print(
            json.dumps(
                threshold_config,
                indent=4
            )
        )

        raise ValueError(
            f'Unable to verify saved thresholds: {run_id}'
        )

    threshold_key, saved_thresholds = (
        valid_candidates[0]
    )

    for key, candidate in valid_candidates:

        assert np.array_equal(
            saved_thresholds,
            candidate
        ), f'Conflicting saved thresholds: {run_id}'

    return threshold_key, saved_thresholds


# Verify every registered experiment and resolve artifact paths

run_configs = {}
run_artifacts = {}

verification_records = []
threshold_format_records = []

for run_id in expected_run_ids:

    record = registry_df.loc[
        registry_df['run_id'] == run_id
    ].iloc[0]

    config_path = PROJECT_ROOT / record['config_path']
    model_path = PROJECT_ROOT / record['model_path']
    results_path = PROJECT_ROOT / record['results_path']

    assert config_path.is_file()
    assert model_path.is_file()
    assert results_path.is_dir()

    with open(config_path, 'r') as file:
        run_config = json.load(file)

    assert run_config['run_id'] == run_id
    assert run_config['labels'] == LABELS
    assert run_config['seed'] == SEED

    assert run_config['feature_representation'] == (
        record['feature_representation']
    )

    assert run_config['evaluation_module_sha256'] == (
        evaluation_hash
    )

    assert run_config['feature_manifest_sha256'] == (
        feature_manifest_hash
    )

    assert not run_config['official_test_used']


    # Resolve validation prediction files

    prediction_candidates = [
        results_path / 'predictions'
        / 'validation_predictions.parquet',

        results_path / 'predictions'
        / f'{run_id}_validation_predictions.parquet'
    ]

    if run_config.get('predictions_path'):

        prediction_candidates.append(
            PROJECT_ROOT / run_config['predictions_path']
        )

    prediction_paths = list({
        path.resolve()
        for path in prediction_candidates
        if path.is_file()
    })

    assert len(prediction_paths) == 1, (
        f'Prediction resolution failed: {run_id}'
    )

    predictions_path = prediction_paths[0]


    # Resolve the saved threshold configuration

    threshold_candidates = [
        config_path.parent / f'{run_id}_thresholds.json'
    ]

    if run_config.get('threshold_path'):

        threshold_candidates.append(
            PROJECT_ROOT / run_config['threshold_path']
        )

    threshold_paths = list({
        path.resolve()
        for path in threshold_candidates
        if path.is_file()
    })

    assert len(threshold_paths) == 1, (
        f'Threshold resolution failed: {run_id}'
    )

    threshold_path = threshold_paths[0]

    original_threshold_hash = calculate_sha256(
        threshold_path
    )

    with open(threshold_path, 'r') as file:
        threshold_config = json.load(file)

    assert threshold_config['run_id'] == run_id
    assert threshold_config['labels'] == LABELS
    assert not threshold_config['official_test_used']

    assert threshold_config.get(
        'selection_partition'
    ) == 'validation'

    assert threshold_config.get(
        'optimization_metric'
    ) == 'per_label_f1'

    assert threshold_config.get(
        'tie_breaking'
    ) == 'highest_threshold'


    # Verify historical threshold comparison conventions

    comparison_operator = threshold_config.get(
        'comparison_operator'
    )

    assert comparison_operator in [
        '>=',
        'greater_than_or_equal'
    ], (
        f'Unsupported threshold comparison: {run_id}'
    )


    # Resolve optimized thresholds from the original JSON

    threshold_key, saved_thresholds = (
        resolve_saved_thresholds(
            threshold_config,
            predictions_path,
            run_id
        )
    )

    assert saved_thresholds.shape == (
        len(LABELS),
    )

    assert np.isfinite(saved_thresholds).all()

    expected_default_threshold = (
        0.0 if record['model'] == 'Linear SVM'
        else 0.5
    )

    assert np.isclose(
        float(threshold_config['default_threshold']),
        expected_default_threshold,
        rtol=0,
        atol=1e-12
    )

    if record['model'] != 'Linear SVM':

        assert (saved_thresholds >= 0).all()
        assert (saved_thresholds <= 1).all()


    # Normalize threshold metadata in memory only

    normalized_threshold_config = {
        **threshold_config,

        'optimized_thresholds': {
            label: float(value)
            for label, value in zip(
                LABELS,
                saved_thresholds
            )
        },

        'comparison_operator': '>=',
        'original_threshold_key': threshold_key
    }

    assert calculate_sha256(
        threshold_path
    ) == original_threshold_hash

    threshold_format_records.append({
        'Run ID': run_id,
        'Original Threshold Key': threshold_key,
        'Original Comparison Operator':
            comparison_operator,
        'Normalized Comparison Operator': '>=',
        'Threshold Count': len(saved_thresholds),
        'Default Threshold':
            expected_default_threshold,
        'Saved Predictions Reproduced': True,
        'Original JSON Unchanged': True,
        'Verification': 'Passed'
    })


    # Verify previously recorded artifact fingerprints

    manifest_candidates = [
        results_path / 'tables' / 'artifact_manifest.csv',
        results_path / 'tables' / 'nb_artifact_manifest.csv'
    ]

    available_manifest_paths = [
        path for path in manifest_candidates
        if path.is_file()
    ]

    verified_fingerprints = 0

    manifest_status = 'Standard manifest unavailable'

    if available_manifest_paths:

        manifest_path = available_manifest_paths[0]

        artifact_manifest = pd.read_csv(
            manifest_path
        )

        required_columns = {
            'Path',
            'SHA256',
            'Size Bytes'
        }

        if required_columns.issubset(
            artifact_manifest.columns
        ):

            for _, artifact in artifact_manifest.iterrows():

                artifact_path = (
                    PROJECT_ROOT / artifact['Path']
                )

                assert artifact_path.is_file()

                assert artifact_path.stat().st_size == (
                    artifact['Size Bytes']
                )

                assert calculate_sha256(
                    artifact_path
                ) == artifact['SHA256'], (
                    f'Fingerprint mismatch: {artifact_path}'
                )

                verified_fingerprints += 1

            manifest_status = 'Verified'

        else:

            manifest_status = 'Nonstandard manifest format'


    # Preserve resolved source artifact references

    run_configs[run_id] = run_config

    run_artifacts[run_id] = {
        'config_path': config_path,
        'model_path': model_path,
        'results_path': results_path,
        'predictions_path': predictions_path,
        'threshold_path': threshold_path,
        'threshold_config': normalized_threshold_config,
        'original_threshold_config': threshold_config,
        'optimized_thresholds': saved_thresholds.copy(),
        'model_sha256': calculate_sha256(model_path),
        'prediction_sha256': calculate_sha256(predictions_path),
        'config_sha256': calculate_sha256(config_path),
        'threshold_sha256': original_threshold_hash
    }

    verification_records.append({
        'Run ID': run_id,
        'Model': record['model'],
        'Representation': record['feature_representation'],
        'Configuration Available': True,
        'Model Available': True,
        'Predictions Available': True,
        'Thresholds Available': True,
        'Threshold Predictions Verified': True,
        'Evaluation Module Verified': True,
        'Feature Manifest Verified': True,
        'Previous Manifest Status': manifest_status,
        'Previous Fingerprints Verified': verified_fingerprints,
        'Official Test Used': False,
        'Verification': 'Passed'
    })

    print(
        f'{run_id}: Verified '
        f'(threshold key: {threshold_key})'
    )


# Save experiment verification results

experiment_verification_df = pd.DataFrame(
    verification_records
)

assert len(experiment_verification_df) == 11

assert experiment_verification_df[
    'Verification'
].eq('Passed').all()

experiment_verification_df.to_csv(
    TABLES_DIR / 'classical_experiment_verification.csv',
    index=False
)

print("\nClassical experiment verification")

display(experiment_verification_df)


# Save threshold format compatibility evidence

threshold_format_df = pd.DataFrame(
    threshold_format_records
)

assert len(threshold_format_df) == 11

assert threshold_format_df[
    'Saved Predictions Reproduced'
].all()

assert threshold_format_df[
    'Original JSON Unchanged'
].all()

threshold_format_df.to_csv(
    TABLES_DIR / 'classical_threshold_format_verification.csv',
    index=False
)

print("\nThreshold configuration compatibility")

display(threshold_format_df)


# Verify shared dataset consistency

training_hashes = {
    item['training_dataset_sha256']
    for item in run_configs.values()
}

validation_hashes = {
    item['validation_dataset_sha256']
    for item in run_configs.values()
}

assert len(training_hashes) == 1
assert len(validation_hashes) == 1

EXPECTED_TRAINING_HASH = next(iter(training_hashes))
EXPECTED_VALIDATION_HASH = next(iter(validation_hashes))

dataset_manifest = pd.read_csv(
    SPLITS_DIR / 'prepared_dataset_manifest.csv'
)

dataset_checks = []

for partition, expected_hash, expected_rows in [
    (
        'train_representative',
        EXPECTED_TRAINING_HASH,
        59997
    ),
    (
        'validation',
        EXPECTED_VALIDATION_HASH,
        31911
    )
]:

    records = dataset_manifest.loc[
        dataset_manifest['Partition'] == partition
    ]

    assert len(records) == 1

    record = records.iloc[0]

    dataset_path = PROJECT_ROOT / record['Dataset Path']

    assert dataset_path.is_file()
    assert int(record['Rows']) == expected_rows

    actual_hash = calculate_sha256(dataset_path)

    assert actual_hash == expected_hash
    assert actual_hash == record['SHA256']

    dataset_checks.append({
        'Partition': partition,
        'Rows': expected_rows,
        'SHA256': actual_hash,
        'All Experiments Match': True,
        'Verification': 'Passed'
    })

dataset_consistency_df = pd.DataFrame(
    dataset_checks
)

dataset_consistency_df.to_csv(
    TABLES_DIR / 'classical_dataset_consistency.csv',
    index=False
)

print("\nShared dataset consistency")

display(dataset_consistency_df)


# Verify completed experiment coverage

coverage_df = pd.crosstab(
    registry_df['model'],
    registry_df['feature_representation']
)

assert coverage_df.to_numpy().sum() == 11

expected_coverage = {
    'Multinomial Naive Bayes': 1,
    'Logistic Regression': 4,
    'Linear SVM': 4,
    'Decision Tree': 1,
    'Random Forest': 1
}

assert registry_df[
    'model'
].value_counts().to_dict() == expected_coverage

print("\nCompleted experiment coverage")

display(coverage_df)

coverage_df.to_csv(
    TABLES_DIR / 'classical_experiment_coverage.csv'
)


# Verify previous methodological decisions

decisions_df = pd.read_csv(
    DECISION_LOG_PATH,
    dtype=str,
    keep_default_na=False
)

assert decisions_df['decision_id'].is_unique

assert set(decisions_df['decision_id']) == {
    f'D{i:03d}' for i in range(1, 48)
}

assert len(decisions_df) == 47


# Record the current environment

environment_df = pd.DataFrame([
    {
        'Package': 'Python',
        'Version': platform.python_version()
    },
    {
        'Package': 'NumPy',
        'Version': np.__version__
    },
    {
        'Package': 'Pandas',
        'Version': pd.__version__
    },
    {
        'Package': 'SciPy',
        'Version': scipy.__version__
    },
    {
        'Package': 'Scikit-learn',
        'Version': sklearn.__version__
    },
    {
        'Package': 'Joblib',
        'Version': joblib.__version__
    }
])

environment_df.to_csv(
    TABLES_DIR / 'environment_versions.csv',
    index=False
)

reference_environment_df = pd.read_csv(
    CLASSICAL_DIR / '00_feature_extraction'
    / 'results' / 'tables'
    / 'environment_versions.csv'
)

environment_comparison_df = environment_df.merge(
    reference_environment_df,
    on='Package',
    suffixes=('_Current', '_Reference'),
    validate='one_to_one'
)

assert len(environment_comparison_df) == len(
    environment_df
)

environment_comparison_df['Version Match'] = (
    environment_comparison_df['Version_Current']
    == environment_comparison_df['Version_Reference']
)

assert environment_comparison_df['Version Match'].all()

environment_comparison_df.to_csv(
    TABLES_DIR / 'environment_compatibility.csv',
    index=False
)

print("\nEnvironment compatibility")

display(environment_comparison_df)


# Configure consistent visualization settings

sns.set_theme(style='whitegrid')

MODEL_COLORS = {
    'Multinomial Naive Bayes': '#94A3B8',
    'Logistic Regression': '#4C78A8',
    'Linear SVM': '#54A24B',
    'Decision Tree': '#F58518',
    'Random Forest': '#B279A2'
}

LABEL_COLORS = {
    'toxic': '#4C78A8',
    'severe_toxic': '#F58518',
    'obscene': '#54A24B',
    'threat': '#E45756',
    'insult': '#B279A2',
    'identity_hate': '#72B7B2'
}

REPRESENTATION_COLORS = {
    'word_tfidf': '#4C78A8',
    'character_tfidf': '#F58518',
    'combined_tfidf': '#54A24B'
}


# Build the consolidation configuration

comparison_config = {
    'notebook': NOTEBOOK_DIR.name,
    'purpose': 'Consolidated classical ML comparison',
    'labels': LABELS,
    'seed': SEED,
    'feature_budget': FEATURE_BUDGET,
    'training_samples': 59997,
    'validation_samples': 31911,
    'expected_runs': expected_run_ids,
    'expected_model_families': expected_coverage,
    'primary_selection_metric': 'macro_average_precision',
    'secondary_selection_metric': 'optimized_macro_f1',
    'training_dataset_sha256': EXPECTED_TRAINING_HASH,
    'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,
    'evaluation_module_sha256': evaluation_hash,
    'feature_manifest_sha256': feature_manifest_hash,
    'source_registry_sha256': registry_original_hash,
    'source_decision_log_sha256': decision_log_original_hash,
    'threshold_compatibility': {
        'supported_primary_keys': [
            'thresholds',
            'optimized_thresholds'
        ],
        'normalized_key': 'optimized_thresholds',
        'comparison_operator': '>=',
        'normalization_scope': 'in_memory_only',
        'original_artifacts_modified': False,
        'saved_binary_predictions_verified': True
    },
    'model_training_planned': False,
    'official_test_used': False,
    'final_project_model_frozen': False
}

COMPARISON_CONFIG_PATH = (
    RUN_CONFIGS_DIR / 'classical_comparison_config.json'
)

with open(COMPARISON_CONFIG_PATH, 'w') as file:
    json.dump(comparison_config, file, indent=4)

with open(COMPARISON_CONFIG_PATH, 'r') as file:
    assert json.load(file) == comparison_config


# Save fingerprints of resolved source artifacts

source_artifact_records = []

for run_id, artifacts in run_artifacts.items():

    for artifact_type in [
        'model',
        'prediction',
        'config',
        'threshold'
    ]:

        path_key = (
            'predictions_path'
            if artifact_type == 'prediction'
            else f'{artifact_type}_path'
        )

        path = artifacts[path_key]

        expected_hash = artifacts[
            f'{artifact_type}_sha256'
        ]

        assert path.is_file()
        assert calculate_sha256(path) == expected_hash

        source_artifact_records.append({
            'Run ID': run_id,
            'Artifact Type': artifact_type,
            'Path': str(path.relative_to(PROJECT_ROOT)),
            'Size Bytes': path.stat().st_size,
            'SHA256': expected_hash
        })

source_artifact_df = pd.DataFrame(
    source_artifact_records
)

assert len(source_artifact_df) == 44

assert not source_artifact_df.duplicated(
    subset=['Run ID', 'Artifact Type']
).any()

source_artifact_df.to_csv(
    TABLES_DIR / 'classical_source_artifact_manifest.csv',
    index=False
)


# Verify that all original artifacts remain unchanged

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)

for run_id, artifacts in run_artifacts.items():

    assert calculate_sha256(
        artifacts['threshold_path']
    ) == artifacts['threshold_sha256']

assert calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
) == evaluation_hash

assert calculate_sha256(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
) == feature_manifest_hash


# Initialize consolidated result storage

classical_results = {}

gc.collect()


# Summarize initialization

print("\nClassical comparison initialization summary")

print("Notebook:", NOTEBOOK_DIR.name)

print("Registered experiments verified:",
      len(experiment_verification_df))

print("Model families:",
      registry_df['model'].nunique())

print("Feature representations:",
      registry_df['feature_representation'].nunique())

print("Training samples:", 59997)
print("Validation samples:", 31911)
print("Feature budget:", FEATURE_BUDGET)

print("Threshold configurations verified:",
      len(threshold_format_df))

print("Threshold prediction reconstruction:",
      threshold_format_df[
          'Saved Predictions Reproduced'
      ].sum(),
      "/", len(threshold_format_df))

print("Historical threshold formats:",
      threshold_format_df[
          'Original Threshold Key'
      ].value_counts().to_dict())

print("Previous artifact fingerprints verified:",
      experiment_verification_df[
          'Previous Fingerprints Verified'
      ].sum())

print("Source artifact fingerprints recorded:",
      len(source_artifact_df))

print("Standard manifests verified:",
      experiment_verification_df[
          'Previous Manifest Status'
      ].eq('Verified').sum())

print("Matching environment packages:",
      environment_comparison_df['Version Match'].sum(),
      "/", len(environment_df))

print("Previous methodological decisions:",
      len(decisions_df))

print("Shared dataset fingerprints: Verified")
print("Shared evaluation module: Verified")
print("Original threshold JSON files: Unchanged")
print("Original experiment registry: Unchanged")
print("Original decision log: Unchanged")
print("Comparison configuration: Saved and verified")
print("Full metric reconstruction: Pending")
print("Model training performed: No")
print("Official test dataset: Not loaded")

print("\nClassical comparison initialized successfully.")

In [ ]:
# Load the verified validation dataset

validation_record = dataset_manifest.loc[
    dataset_manifest['Partition'] == 'validation'
].iloc[0]

validation_path = (
    PROJECT_ROOT / validation_record['Dataset Path']
)

assert calculate_sha256(validation_path) == (
    EXPECTED_VALIDATION_HASH
)

val_df = pd.read_parquet(validation_path)

assert len(val_df) == 31911
assert val_df['id'].is_unique
assert val_df['group_id'].notna().all()
assert val_df[LABELS].isin([0, 1]).all().all()

y_val = val_df[LABELS].to_numpy(dtype=np.int8)

assert y_val.shape == (31911, 6)

assert (y_val.sum(axis=0) > 0).all()
assert (y_val.sum(axis=0) < len(y_val)).all()


# Record validation label support

validation_support_df = pd.DataFrame({
    'Label': LABELS,
    'Positive Support': y_val.sum(axis=0),
    'Negative Support': len(y_val) - y_val.sum(axis=0),
    'Prevalence (%)': 100 * y_val.mean(axis=0)
})

print("Validation label support")

display(validation_support_df.round(6))

validation_support_df.to_csv(
    TABLES_DIR / 'classical_validation_label_support.csv',
    index=False
)


# Initialize consolidated evaluation records

classical_results = {}

overall_records = []
per_label_records = []
threshold_records = []
reconstruction_records = []
computational_records = []


# Reconstruct all completed experiments

for run_id in expected_run_ids:

    record = registry_df.loc[
        registry_df['run_id'] == run_id
    ].iloc[0]

    current_config = run_configs[run_id]
    artifacts = run_artifacts[run_id]

    model_name = record['model']
    representation = record['feature_representation']

    class_weight = (
        'balanced'
        if '_BALANCED_' in run_id
        else 'None'
    )

    score_type = (
        'decision_margin'
        if model_name == 'Linear SVM'
        else 'positive_class_probability'
    )


    # Verify source artifacts remain unchanged

    for path_key, hash_key in [
        ('model_path', 'model_sha256'),
        ('predictions_path', 'prediction_sha256'),
        ('config_path', 'config_sha256'),
        ('threshold_path', 'threshold_sha256')
    ]:

        assert calculate_sha256(
            artifacts[path_key]
        ) == artifacts[hash_key]

    predictions = pd.read_parquet(
        artifacts['predictions_path']
    )

    assert len(predictions) == len(val_df)
    assert predictions['id'].is_unique

    assert set(predictions['id']) == set(val_df['id'])


    # Align predictions using validated comment IDs

    original_order_matched = (
        predictions['id'].tolist()
        == val_df['id'].tolist()
    )

    predictions = (
        predictions.set_index('id')
        .loc[val_df['id'].tolist()]
        .reset_index()
    )

    assert predictions['id'].tolist() == (
        val_df['id'].tolist()
    )

    targets = predictions[[
        f'{label}_true' for label in LABELS
    ]].to_numpy()

    scores = predictions[[
        f'{label}_score' for label in LABELS
    ]].to_numpy(dtype=np.float64)

    saved_default_predictions = predictions[[
        f'{label}_pred_default' for label in LABELS
    ]].to_numpy()

    saved_optimized_predictions = predictions[[
        f'{label}_pred_optimized' for label in LABELS
    ]].to_numpy()

    assert np.array_equal(targets, y_val)
    assert scores.shape == y_val.shape
    assert np.isfinite(scores).all()

    assert np.isin(
        saved_default_predictions, [0, 1]
    ).all()

    assert np.isin(
        saved_optimized_predictions, [0, 1]
    ).all()

    if score_type == 'positive_class_probability':

        assert (scores >= 0).all()
        assert (scores <= 1).all()


    # Load original default and optimized thresholds

    threshold_config = artifacts['threshold_config']

    default_threshold = float(
        threshold_config['default_threshold']
    )

    optimized_thresholds = artifacts[
        'optimized_thresholds'
    ].copy()

    assert optimized_thresholds.shape == (
        len(LABELS),
    )

    assert np.isfinite(optimized_thresholds).all()

    assert threshold_config[
        'comparison_operator'
    ] == '>='

    if model_name == 'Linear SVM':
        assert default_threshold == 0.0
    else:
        assert default_threshold == 0.5


    # Reconstruct the original binary predictions

    reconstructed_default = apply_thresholds(
        scores,
        default_threshold
    )

    reconstructed_optimized = apply_thresholds(
        scores,
        optimized_thresholds
    )

    assert np.array_equal(
        reconstructed_default,
        saved_default_predictions
    )

    assert np.array_equal(
        reconstructed_optimized,
        saved_optimized_predictions
    )


    # Reconstruct evaluation metrics

    default_metrics, default_per_label = (
        evaluate_multilabel(
            targets,
            scores,
            LABELS,
            thresholds=default_threshold
        )
    )

    optimized_metrics, optimized_per_label = (
        evaluate_multilabel(
            targets,
            scores,
            LABELS,
            thresholds=optimized_thresholds
        )
    )

    assert default_per_label['Label'].tolist() == LABELS
    assert optimized_per_label['Label'].tolist() == LABELS

    assert np.array_equal(
        optimized_per_label['Positive Support'].to_numpy(),
        y_val.sum(axis=0)
    )

    assert np.array_equal(
        optimized_per_label['Negative Support'].to_numpy(),
        len(y_val) - y_val.sum(axis=0)
    )


    # Independently verify original threshold optimality

    recomputed_threshold_df = optimize_thresholds(
        targets,
        scores,
        LABELS
    )

    assert recomputed_threshold_df[
        'Label'
    ].tolist() == LABELS

    recomputed_thresholds = recomputed_threshold_df[
        'Optimal Threshold'
    ].to_numpy(dtype=np.float64)

    threshold_difference = float(
        np.max(
            np.abs(
                optimized_thresholds
                - recomputed_thresholds
            )
        )
    )

    np.testing.assert_allclose(
        optimized_thresholds,
        recomputed_thresholds,
        rtol=0,
        atol=1e-12
    )

    assert np.array_equal(
        apply_thresholds(scores, recomputed_thresholds),
        saved_optimized_predictions
    )

    np.testing.assert_allclose(
        optimized_per_label['F1'].to_numpy(),
        recomputed_threshold_df['Validation F1'].to_numpy(),
        rtol=0,
        atol=1e-12
    )


    # Verify threshold-independent ranking metrics

    for metric in [
        'Macro AP',
        'Micro AP',
        'Macro ROC-AUC',
        'Micro ROC-AUC'
    ]:

        assert np.isclose(
            default_metrics[metric],
            optimized_metrics[metric],
            rtol=0,
            atol=1e-12
        )

    assert np.all(
        optimized_per_label['F1'].to_numpy()
        >= default_per_label['F1'].to_numpy() - 1e-12
    )


    # Verify reconstructed confusion counts

    for binary_predictions, per_label_metrics in [
        (reconstructed_default, default_per_label),
        (reconstructed_optimized, optimized_per_label)
    ]:

        expected_counts = {
            'TP': (
                (targets == 1)
                & (binary_predictions == 1)
            ).sum(axis=0),

            'TN': (
                (targets == 0)
                & (binary_predictions == 0)
            ).sum(axis=0),

            'FP': (
                (targets == 0)
                & (binary_predictions == 1)
            ).sum(axis=0),

            'FN': (
                (targets == 1)
                & (binary_predictions == 0)
            ).sum(axis=0)
        }

        for column, expected_values in expected_counts.items():

            assert np.array_equal(
                per_label_metrics[column].to_numpy(),
                expected_values
            )


    # Compare reconstructed results with the registry

    registered_ap = float(
        record['validation_macro_ap']
    )

    registered_f1 = float(
        record['validation_macro_f1']
    )

    ap_error = abs(
        optimized_metrics['Macro AP']
        - registered_ap
    )

    f1_error = abs(
        optimized_metrics['Macro F1']
        - registered_f1
    )

    assert ap_error <= 1e-6
    assert f1_error <= 1e-6


    # Recover recorded computational measurements

    training_time = float(
        record['training_time_seconds']
    )

    model_size_mib = (
        artifacts['model_path'].stat().st_size
        / (1024 ** 2)
    )

    summary_candidates = [
        artifacts['results_path']
        / 'tables' / 'run_summary.csv',

        artifacts['results_path']
        / 'tables' / 'nb_run_summary.csv'
    ]

    available_summaries = [
        path for path in summary_candidates
        if path.is_file()
    ]

    prediction_time = np.nan
    prediction_time_source = 'Not available'

    if len(available_summaries) == 1:

        source_summary_df = pd.read_csv(
            available_summaries[0]
        )

        if len(source_summary_df) == 1:

            source_summary = source_summary_df.iloc[0]

            for column in [
                'Validation Prediction Time (s)',
                'Prediction Time (s)'
            ]:

                if column in source_summary.index:

                    value = pd.to_numeric(
                        source_summary[column],
                        errors='coerce'
                    )

                    if pd.notna(value) and value >= 0:

                        prediction_time = float(value)

                        prediction_time_source = str(
                            available_summaries[0]
                            .relative_to(PROJECT_ROOT)
                        )

                        break


    # Build consolidated overall metrics

    overall_record = {
        'Run ID': run_id,
        'Model': model_name,
        'Representation': representation,
        'Class Weight': class_weight,
        'Score Type': score_type,
        'Training Samples': int(record['training_samples']),
        'Validation Samples': len(y_val),
        'Features': FEATURE_BUDGET,
        'Seed': SEED,
        'Macro AP': optimized_metrics['Macro AP'],
        'Micro AP': optimized_metrics['Micro AP'],
        'Macro ROC-AUC': optimized_metrics['Macro ROC-AUC'],
        'Micro ROC-AUC': optimized_metrics['Micro ROC-AUC']
    }

    threshold_dependent_metrics = [
        'Macro Precision',
        'Macro Recall',
        'Macro F1',
        'Micro F1',
        'Hamming Loss',
        'Exact Match Accuracy'
    ]

    for metric in threshold_dependent_metrics:

        overall_record[f'Default {metric}'] = (
            default_metrics[metric]
        )

        overall_record[f'Optimized {metric}'] = (
            optimized_metrics[metric]
        )

    overall_record.update({
        'Macro F1 Improvement': (
            optimized_metrics['Macro F1']
            - default_metrics['Macro F1']
        ),

        'Micro F1 Improvement': (
            optimized_metrics['Micro F1']
            - default_metrics['Micro F1']
        ),

        'Training Time (s)': training_time,
        'Validation Prediction Time (s)': prediction_time,
        'Model Size (MiB)': model_size_mib,
        'Official Test Used': False,
        'Verification': 'Passed'
    })

    overall_records.append(overall_record)


    # Consolidate individual label metrics

    for threshold_type, table in [
        ('Default', default_per_label),
        ('Optimized', optimized_per_label)
    ]:

        label_table = table.copy()

        label_table.insert(
            0, 'Threshold Type', threshold_type
        )

        label_table.insert(
            0, 'Score Type', score_type
        )

        label_table.insert(
            0, 'Class Weight', class_weight
        )

        label_table.insert(
            0, 'Representation', representation
        )

        label_table.insert(
            0, 'Model', model_name
        )

        label_table.insert(
            0, 'Run ID', run_id
        )

        per_label_records.append(label_table)


    # Consolidate threshold values and effects

    for i, label in enumerate(LABELS):

        default_row = default_per_label.iloc[i]
        optimized_row = optimized_per_label.iloc[i]

        threshold_records.append({
            'Run ID': run_id,
            'Model': model_name,
            'Representation': representation,
            'Class Weight': class_weight,
            'Score Type': score_type,
            'Label': label,
            'Default Threshold': default_threshold,
            'Optimized Threshold': float(
                optimized_thresholds[i]
            ),
            'Default Precision': default_row['Precision'],
            'Optimized Precision': optimized_row['Precision'],
            'Default Recall': default_row['Recall'],
            'Optimized Recall': optimized_row['Recall'],
            'Default F1': default_row['F1'],
            'Optimized F1': optimized_row['F1'],
            'F1 Improvement': (
                optimized_row['F1'] - default_row['F1']
            ),
            'Default Predicted Positives':
                default_row['Predicted Positives'],
            'Optimized Predicted Positives':
                optimized_row['Predicted Positives'],
            'Actual Positives':
                optimized_row['Positive Support']
        })


    # Record computational measurements and limitations

    computational_records.append({
        'Run ID': run_id,
        'Model': model_name,
        'Representation': representation,
        'Class Weight': class_weight,
        'Training Samples': int(record['training_samples']),
        'Features': FEATURE_BUDGET,
        'Training Time (s)': training_time,
        'Validation Prediction Time (s)': prediction_time,
        'Prediction Time Source': prediction_time_source,
        'Model Size (MiB)': model_size_mib,
        'Vectorizer Size Included': False,
        'Repeated Runtime Measurements': False
    })


    # Record independent reconstruction evidence

    reconstruction_records.append({
        'Run ID': run_id,
        'Validation Rows': len(targets),
        'Label Count': len(LABELS),
        'Original ID Order Matched': original_order_matched,
        'ID Alignment Verified': True,
        'Target Labels Verified': True,
        'Default Predictions Reproduced': True,
        'Optimized Predictions Reproduced': True,
        'Threshold Optimality Verified': True,
        'Confusion Counts Verified': True,
        'Ranking Metrics Invariant': True,
        'Maximum Threshold Difference': threshold_difference,
        'Registry Macro AP': registered_ap,
        'Reconstructed Macro AP': optimized_metrics['Macro AP'],
        'Macro AP Reconstruction Error': ap_error,
        'Registry Macro F1': registered_f1,
        'Reconstructed Macro F1': optimized_metrics['Macro F1'],
        'Macro F1 Reconstruction Error': f1_error,
        'Source Fingerprints Verified': True,
        'Official Test Used': False,
        'Verification': 'Passed'
    })


    # Retain verified results for later analysis

    classical_results[run_id] = {
        'model': model_name,
        'representation': representation,
        'class_weight': class_weight,
        'score_type': score_type,
        'summary': overall_record,
        'default_metrics': default_metrics,
        'optimized_metrics': optimized_metrics,
        'default_per_label': default_per_label,
        'optimized_per_label': optimized_per_label,
        'default_threshold': default_threshold,
        'optimized_thresholds': optimized_thresholds,
        'thresholds': recomputed_threshold_df,
        'scores': scores,
        'default_predictions': reconstructed_default,
        'optimized_predictions': reconstructed_optimized
    }

    print(
        f"{run_id}: Passed | "
        f"AP = {optimized_metrics['Macro AP']:.6f} | "
        f"F1 = {optimized_metrics['Macro F1']:.6f}"
    )


# Construct consolidated results tables

classical_overall_df = pd.DataFrame(
    overall_records
)

classical_per_label_df = pd.concat(
    per_label_records,
    ignore_index=True
)

classical_threshold_df = pd.DataFrame(
    threshold_records
)

classical_reconstruction_df = pd.DataFrame(
    reconstruction_records
)

classical_computational_df = pd.DataFrame(
    computational_records
)


# Verify complete experimental coverage

assert len(classical_results) == 11
assert len(classical_overall_df) == 11
assert len(classical_per_label_df) == 11 * 6 * 2
assert len(classical_threshold_df) == 11 * 6
assert len(classical_reconstruction_df) == 11
assert len(classical_computational_df) == 11

assert not classical_per_label_df.duplicated([
    'Run ID', 'Label', 'Threshold Type'
]).any()

assert not classical_threshold_df.duplicated([
    'Run ID', 'Label'
]).any()

assert classical_reconstruction_df[
    'Verification'
].eq('Passed').all()

assert not classical_overall_df[
    'Official Test Used'
].any()

assert classical_overall_df[
    'Training Samples'
].eq(59997).all()

assert classical_overall_df[
    'Features'
].eq(FEATURE_BUDGET).all()


# Rank all configurations by the predefined selection policy

classical_ranking_df = (
    classical_overall_df.sort_values(
        [
            'Macro AP',
            'Optimized Macro F1',
            'Run ID'
        ],
        ascending=[False, False, True]
    )
    .reset_index(drop=True)
)

classical_ranking_df.insert(
    0,
    'Rank',
    np.arange(1, len(classical_ranking_df) + 1)
)

print("\nReconstructed classical validation leaderboard")

display(
    classical_ranking_df[[
        'Rank',
        'Run ID',
        'Macro AP',
        'Macro ROC-AUC',
        'Default Macro F1',
        'Optimized Macro F1',
        'Optimized Micro F1',
        'Training Time (s)',
        'Model Size (MiB)'
    ]].round(6)
)


# Display reconstruction accuracy

print("\nIndependent metric reconstruction verification")

display(
    classical_reconstruction_df[[
        'Run ID',
        'Maximum Threshold Difference',
        'Macro AP Reconstruction Error',
        'Macro F1 Reconstruction Error',
        'Verification'
    ]]
)


# Save all consolidated numerical evidence

consolidated_tables = {
    'classical_all_run_metrics.csv':
        classical_overall_df,

    'classical_all_per_label_metrics.csv':
        classical_per_label_df,

    'classical_all_thresholds.csv':
        classical_threshold_df,

    'classical_metric_reconstruction.csv':
        classical_reconstruction_df,

    'classical_computational_measurements.csv':
        classical_computational_df,

    'classical_validation_ranking.csv':
        classical_ranking_df
}

for filename, table in consolidated_tables.items():

    path = TABLES_DIR / filename

    table.to_csv(
        path,
        index=False
    )

    assert path.is_file()
    assert path.stat().st_size > 0

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape


# Verify original experiment artifacts remain unchanged

for _, artifact in source_artifact_df.iterrows():

    path = PROJECT_ROOT / artifact['Path']

    assert path.is_file()

    assert calculate_sha256(path) == artifact['SHA256']

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)

assert calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
) == evaluation_hash

assert calculate_sha256(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
) == feature_manifest_hash


# Save reconstruction metadata

reconstruction_summary = {
    'experiments_reconstructed': len(classical_results),
    'validation_samples': len(y_val),
    'labels': LABELS,
    'overall_metric_rows': len(classical_overall_df),
    'per_label_metric_rows': len(classical_per_label_df),
    'threshold_rows': len(classical_threshold_df),
    'maximum_threshold_difference': float(
        classical_reconstruction_df[
            'Maximum Threshold Difference'
        ].max()
    ),
    'maximum_macro_ap_reconstruction_error': float(
        classical_reconstruction_df[
            'Macro AP Reconstruction Error'
        ].max()
    ),
    'maximum_macro_f1_reconstruction_error': float(
        classical_reconstruction_df[
            'Macro F1 Reconstruction Error'
        ].max()
    ),
    'leading_run_id': str(
        classical_ranking_df.loc[0, 'Run ID']
    ),
    'leading_validation_macro_ap': float(
        classical_ranking_df.loc[0, 'Macro AP']
    ),
    'prediction_time_available_runs': int(
        classical_computational_df[
            'Validation Prediction Time (s)'
        ].notna().sum()
    ),
    'source_artifacts_unchanged': True,
    'original_thresholds_modified': False,
    'model_training_performed': False,
    'official_test_used': False,
    'final_project_model_frozen': False
}

with open(
    LOGS_DIR / 'classical_reconstruction_summary.json',
    'w'
) as file:

    json.dump(
        reconstruction_summary,
        file,
        indent=4
    )

gc.collect()


# Summarize completed metric reconstruction

print("\nClassical metric reconstruction summary")

print("Experiments reconstructed:",
      len(classical_results), "/ 11")

print("Validation samples:", len(y_val))
print("Overall metric rows:", len(classical_overall_df))
print("Per-label metric rows:", len(classical_per_label_df))
print("Threshold records:", len(classical_threshold_df))

print("Maximum threshold reconstruction difference:",
      reconstruction_summary['maximum_threshold_difference'])

print("Maximum Macro AP reconstruction error:",
      reconstruction_summary[
          'maximum_macro_ap_reconstruction_error'
      ])

print("Maximum Macro F1 reconstruction error:",
      reconstruction_summary[
          'maximum_macro_f1_reconstruction_error'
      ])

print("Recorded prediction times available:",
      reconstruction_summary['prediction_time_available_runs'],
      "/ 11")

print("Current leading configuration:",
      reconstruction_summary['leading_run_id'])

print("Leading validation Macro AP:",
      round(
          reconstruction_summary[
              'leading_validation_macro_ap'
          ],
          6
      ))

print("Consolidated tables saved:",
      len(consolidated_tables) + 1)

print("Original source fingerprints: Unchanged")
print("Original experiment registry: Unchanged")
print("Original thresholds: Unchanged")
print("Model training performed: No")
print("Official test dataset: Not loaded")
print("Final project model frozen: No")

print("\nClassical metric reconstruction completed successfully.")

In [ ]:
# Select the five matched word-feature baselines

common_run_ids = [
    'NB_WORD_BASELINE_S42',
    'LR_WORD_BASELINE_S42',
    'SVM_WORD_BASELINE_S42',
    'DT_WORD_BASELINE_S42',
    'RF_WORD_BASELINE_S42'
]

common_baseline_df = (
    classical_overall_df
    .set_index('Run ID')
    .loc[common_run_ids]
    .reset_index()
    .copy()
)

assert len(common_baseline_df) == 5
assert common_baseline_df['Model'].nunique() == 5
assert common_baseline_df['Representation'].eq('word_tfidf').all()
assert common_baseline_df['Class Weight'].eq('None').all()
assert common_baseline_df['Training Samples'].eq(59997).all()
assert common_baseline_df['Validation Samples'].eq(31911).all()
assert common_baseline_df['Features'].eq(FEATURE_BUDGET).all()
assert common_baseline_df['Seed'].eq(SEED).all()
assert not common_baseline_df['Official Test Used'].any()


# Calculate the theoretical constant-score AP baseline

constant_score_macro_ap = float(
    y_val.mean(axis=0).mean()
)

common_baseline_df['Constant Score Macro AP'] = (
    constant_score_macro_ap
)

common_baseline_df['Macro AP Above Constant Baseline'] = (
    common_baseline_df['Macro AP']
    - constant_score_macro_ap
)


# Rank algorithms using the primary selection metric

common_baseline_ranking_df = (
    common_baseline_df.sort_values(
        ['Macro AP', 'Optimized Macro F1', 'Run ID'],
        ascending=[False, False, True]
    )
    .reset_index(drop=True)
)

common_baseline_ranking_df.insert(
    0,
    'Rank',
    np.arange(1, len(common_baseline_ranking_df) + 1)
)

best_word_run_id = common_baseline_ranking_df.loc[
    0, 'Run ID'
]

best_word_macro_ap = float(
    common_baseline_ranking_df.loc[0, 'Macro AP']
)

common_baseline_ranking_df['Macro AP Gap from Best'] = (
    best_word_macro_ap
    - common_baseline_ranking_df['Macro AP']
)

print("Controlled Word TF-IDF algorithm comparison")

display(
    common_baseline_ranking_df[[
        'Rank',
        'Model',
        'Macro AP',
        'Macro ROC-AUC',
        'Default Macro F1',
        'Optimized Macro F1',
        'Optimized Micro F1',
        'Macro AP Gap from Best'
    ]].round(6)
)

common_baseline_ranking_df.to_csv(
    TABLES_DIR / 'classical_common_algorithm_comparison.csv',
    index=False
)


# Extract per-label metrics for the matched configurations

common_per_label_df = classical_per_label_df.loc[
    classical_per_label_df['Run ID'].isin(common_run_ids)
    & classical_per_label_df['Threshold Type'].eq('Optimized')
].copy()

assert len(common_per_label_df) == 5 * len(LABELS)

assert not common_per_label_df.duplicated([
    'Run ID', 'Label'
]).any()

common_per_label_df.to_csv(
    TABLES_DIR / 'classical_common_algorithm_per_label.csv',
    index=False
)


# Construct label-wise comparison matrices

model_order = common_baseline_ranking_df['Model'].tolist()

common_ap_df = (
    common_per_label_df.pivot(
        index='Label',
        columns='Model',
        values='Average Precision'
    )
    .reindex(
        index=LABELS,
        columns=model_order
    )
)

common_f1_df = (
    common_per_label_df.pivot(
        index='Label',
        columns='Model',
        values='F1'
    )
    .reindex(
        index=LABELS,
        columns=model_order
    )
)

assert not common_ap_df.isna().any().any()
assert not common_f1_df.isna().any().any()

print("\nPer-label Average Precision")

display(common_ap_df.round(6))

print("\nPer-label optimized F1")

display(common_f1_df.round(6))

common_ap_df.to_csv(
    TABLES_DIR / 'classical_common_algorithm_ap_matrix.csv'
)

common_f1_df.to_csv(
    TABLES_DIR / 'classical_common_algorithm_f1_matrix.csv'
)


# Identify the leading algorithm for each label

label_winner_records = []

for label in LABELS:

    best_ap = float(common_ap_df.loc[label].max())
    best_f1 = float(common_f1_df.loc[label].max())

    ap_winners = common_ap_df.columns[
        np.isclose(
            common_ap_df.loc[label].to_numpy(),
            best_ap,
            rtol=0,
            atol=1e-12
        )
    ].tolist()

    f1_winners = common_f1_df.columns[
        np.isclose(
            common_f1_df.loc[label].to_numpy(),
            best_f1,
            rtol=0,
            atol=1e-12
        )
    ].tolist()

    label_winner_records.append({
        'Label': label,
        'Validation Positives': int(
            y_val[:, LABELS.index(label)].sum()
        ),
        'Constant Score AP': float(
            y_val[:, LABELS.index(label)].mean()
        ),
        'Best AP Model': ', '.join(ap_winners),
        'Best AP': best_ap,
        'Best F1 Model': ', '.join(f1_winners),
        'Best Optimized F1': best_f1
    })

common_label_winners_df = pd.DataFrame(
    label_winner_records
)

print("\nBest common-feature algorithm by toxicity label")

display(common_label_winners_df.round(6))

common_label_winners_df.to_csv(
    TABLES_DIR / 'classical_common_algorithm_label_winners.csv',
    index=False
)


# Compare performance on the three rarest toxicity labels

rare_labels = [
    'severe_toxic',
    'threat',
    'identity_hate'
]

rare_label_records = []

for run_id in common_run_ids:

    model_results = common_per_label_df.loc[
        common_per_label_df['Run ID'].eq(run_id)
    ]

    rare_results = model_results.loc[
        model_results['Label'].isin(rare_labels)
    ]

    assert len(rare_results) == len(rare_labels)

    rare_label_records.append({
        'Run ID': run_id,
        'Model': classical_results[run_id]['model'],
        'Rare Label Mean AP':
            float(rare_results['Average Precision'].mean()),
        'Rare Label Mean Precision':
            float(rare_results['Precision'].mean()),
        'Rare Label Mean Recall':
            float(rare_results['Recall'].mean()),
        'Rare Label Mean F1':
            float(rare_results['F1'].mean()),
        'Rare Label Total TP':
            int(rare_results['TP'].sum()),
        'Rare Label Total FP':
            int(rare_results['FP'].sum()),
        'Rare Label Total FN':
            int(rare_results['FN'].sum())
    })

common_rare_label_df = (
    pd.DataFrame(rare_label_records)
    .sort_values(
        ['Rare Label Mean AP', 'Rare Label Mean F1'],
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nRare-label performance comparison")

display(common_rare_label_df.round(6))

common_rare_label_df.to_csv(
    TABLES_DIR / 'classical_common_algorithm_rare_labels.csv',
    index=False
)


# Verify consistency with the Random Forest notebook

rf_reference_path = (
    CLASSICAL_DIR / '05_random_forest'
    / 'results' / 'tables'
    / 'rf_common_baseline_comparison.csv'
)

assert rf_reference_path.is_file()

rf_reference_df = pd.read_csv(
    rf_reference_path
)

comparison_metrics = [
    'Macro AP',
    'Macro ROC-AUC',
    'Default Macro F1',
    'Optimized Macro F1',
    'Default Micro F1',
    'Optimized Micro F1'
]

previous_results_check = (
    common_baseline_df[[
        'Run ID', *comparison_metrics
    ]]
    .merge(
        rf_reference_df[[
            'Run ID', *comparison_metrics
        ]],
        on='Run ID',
        suffixes=('_Current', '_Reference'),
        validate='one_to_one'
    )
)

assert len(previous_results_check) == 5

for metric in comparison_metrics:

    np.testing.assert_allclose(
        previous_results_check[f'{metric}_Current'],
        previous_results_check[f'{metric}_Reference'],
        rtol=0,
        atol=1e-12
    )

print("\nPrevious common-feature comparison: Verified")


# Prepare readable model names

short_names = {
    'Multinomial Naive Bayes': 'Naive Bayes',
    'Logistic Regression': 'Logistic Regression',
    'Linear SVM': 'Linear SVM',
    'Decision Tree': 'Decision Tree',
    'Random Forest': 'Random Forest'
}

plot_df = common_baseline_ranking_df.copy()

plot_df['Display Name'] = (
    plot_df['Model'].map(short_names)
)

plot_colors = [
    MODEL_COLORS[model]
    for model in plot_df['Model']
]


# Figure 1: Controlled classical algorithm comparison

fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 6)
)

for ax, metric, title in [
    (
        axes[0],
        'Macro AP',
        'Validation Macro Average Precision'
    ),
    (
        axes[1],
        'Optimized Macro F1',
        'Validation Optimized Macro F1'
    )
]:

    bars = ax.barh(
        plot_df['Display Name'],
        plot_df[metric],
        color=plot_colors,
        height=0.6
    )

    ax.bar_label(
        bars,
        fmt='%.4f',
        padding=5,
        fontsize=10
    )

    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xlabel('Score')
    ax.set_xlim(0, 1.0)
    ax.grid(axis='y', visible=False)

axes[0].axvline(
    constant_score_macro_ap,
    color='gray',
    linestyle='--',
    linewidth=1.5,
    label=f'Constant-score AP = {constant_score_macro_ap:.4f}'
)

axes[0].legend(
    loc='lower right',
    fontsize=9
)

fig.suptitle(
    'Controlled Classical ML Algorithm Comparison',
    fontsize=16
)

fig.text(
    0.5,
    0.02,
    '59,997 training comments | 31,911 validation comments | '
    '50,000 word TF-IDF features | Unweighted configurations',
    ha='center',
    fontsize=10
)

sns.despine()

fig.tight_layout(
    rect=[0, 0.06, 1, 0.93]
)

fig.savefig(
    FIGURES_DIR / 'classical_common_algorithm_comparison.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Figure 2: Per-label performance heatmaps

ap_heatmap = common_ap_df.copy()
f1_heatmap = common_f1_df.copy()

for table in [ap_heatmap, f1_heatmap]:

    table.index = [
        label.replace('_', ' ').title()
        for label in LABELS
    ]

    table.columns = [
        short_names[model]
        for model in table.columns
    ]

fig, axes = plt.subplots(
    1, 2,
    figsize=(17, 7)
)

for ax, table, title in [
    (
        axes[0],
        ap_heatmap,
        'Per-label Average Precision'
    ),
    (
        axes[1],
        f1_heatmap,
        'Per-label Optimized F1'
    )
]:

    sns.heatmap(
        table,
        annot=True,
        fmt='.3f',
        cmap='Blues',
        vmin=0,
        vmax=1,
        linewidths=0.5,
        ax=ax,
        cbar_kws={'label': 'Score'}
    )

    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

fig.suptitle(
    'Label-wise Performance of Common Word TF-IDF Baselines',
    fontsize=16
)

fig.tight_layout(
    rect=[0, 0, 1, 0.94]
)

fig.savefig(
    FIGURES_DIR / 'classical_common_algorithm_per_label_heatmaps.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Record controlled comparison findings

common_algorithm_analysis = {
    'comparison_scope': 'unweighted_word_tfidf',
    'models_compared': len(common_baseline_df),
    'training_samples': 59997,
    'validation_samples': len(y_val),
    'feature_budget': FEATURE_BUDGET,
    'seed': SEED,
    'primary_selection_metric': 'macro_average_precision',
    'constant_score_macro_ap': constant_score_macro_ap,
    'best_common_feature_run': best_word_run_id,
    'best_common_feature_model': str(
        common_baseline_ranking_df.loc[0, 'Model']
    ),
    'best_common_feature_macro_ap': best_word_macro_ap,
    'best_common_feature_optimized_macro_f1': float(
        common_baseline_ranking_df.loc[0, 'Optimized Macro F1']
    ),
    'best_rare_label_ap_model': str(
        common_rare_label_df.loc[0, 'Model']
    ),
    'best_rare_label_mean_ap': float(
        common_rare_label_df.loc[0, 'Rare Label Mean AP']
    ),
    'previous_comparison_reconstructed': True,
    'statistical_significance_established': False,
    'algorithm_hyperparameters_optimized': False,
    'model_training_performed': False,
    'official_test_used': False,
    'final_project_model_frozen': False
}

with open(
    RUN_CONFIGS_DIR / 'classical_common_algorithm_analysis.json',
    'w'
) as file:

    json.dump(
        common_algorithm_analysis,
        file,
        indent=4
    )


# Verify original experiment records remain unchanged

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)


# Summarize the controlled algorithm comparison

print("\nControlled classical algorithm comparison summary")

print("Algorithms compared:", len(common_baseline_df))
print("Matched feature representation: word_tfidf")
print("Matched feature budget:", FEATURE_BUDGET)
print("Training samples:", 59997)
print("Validation samples:", len(y_val))
print("Class weighting: None")

print(
    "Constant-score Macro AP:",
    round(constant_score_macro_ap, 6)
)

print("Best common-feature run:", best_word_run_id)

print(
    "Best common-feature Macro AP:",
    round(best_word_macro_ap, 6)
)

print(
    "Best common-feature optimized Macro F1:",
    round(
        common_baseline_ranking_df.loc[
            0, 'Optimized Macro F1'
        ],
        6
    )
)

print(
    "Best rare-label AP model:",
    common_rare_label_df.loc[0, 'Model']
)

print(
    "Best rare-label mean AP:",
    round(
        common_rare_label_df.loc[0, 'Rare Label Mean AP'],
        6
    )
)

print("Previous comparison consistency: Passed")
print("Analysis tables saved: 6")
print("Research figures generated: 2")
print("Analysis configuration saved: Yes")
print("Original experiment registry: Unchanged")
print("Model training performed: No")
print("Official test dataset: Not loaded")

print("\nControlled classical algorithm comparison completed successfully.")

In [ ]:
# Define matched representation and weighting experiments

linear_run_groups = {
    'Logistic Regression': {
        'Word': 'LR_WORD_BASELINE_S42',
        'Character': 'LR_CHAR_BASELINE_S42',
        'Combined': 'LR_COMBINED_BASELINE_S42',
        'Combined Balanced': 'LR_COMBINED_BALANCED_S42'
    },
    'Linear SVM': {
        'Word': 'SVM_WORD_BASELINE_S42',
        'Character': 'SVM_CHAR_BASELINE_S42',
        'Combined': 'SVM_COMBINED_BASELINE_S42',
        'Combined Balanced': 'SVM_COMBINED_BALANCED_S42'
    }
}

overall_lookup = classical_overall_df.set_index('Run ID')

optimized_label_metrics = classical_per_label_df.loc[
    classical_per_label_df['Threshold Type'].eq('Optimized')
].copy()

assert len(optimized_label_metrics) == 66

rare_labels = [
    'severe_toxic',
    'threat',
    'identity_hate'
]


def get_label_metrics(run_id):

    table = (
        optimized_label_metrics.loc[
            optimized_label_metrics['Run ID'].eq(run_id)
        ]
        .set_index('Label')
        .loc[LABELS]
    )

    assert len(table) == len(LABELS)

    return table


# Consolidate the eight linear classifier configurations

linear_records = []

for model, runs in linear_run_groups.items():

    for configuration, run_id in runs.items():

        row = overall_lookup.loc[run_id]
        label_table = get_label_metrics(run_id)
        rare_table = label_table.loc[rare_labels]

        assert row['Model'] == model
        assert row['Features'] == FEATURE_BUDGET
        assert row['Training Samples'] == 59997
        assert row['Validation Samples'] == 31911
        assert row['Seed'] == SEED

        expected_weight = (
            'balanced'
            if configuration == 'Combined Balanced'
            else 'None'
        )

        assert row['Class Weight'] == expected_weight

        linear_records.append({
            'Model': model,
            'Configuration': configuration,
            'Run ID': run_id,
            'Representation': row['Representation'],
            'Class Weight': row['Class Weight'],
            'Macro AP': row['Macro AP'],
            'Macro ROC-AUC': row['Macro ROC-AUC'],
            'Default Macro F1': row['Default Macro F1'],
            'Optimized Macro F1': row['Optimized Macro F1'],
            'Optimized Micro F1': row['Optimized Micro F1'],
            'Rare Label Mean AP':
                rare_table['Average Precision'].mean(),
            'Rare Label Mean F1':
                rare_table['F1'].mean(),
            'Rare Label Mean Recall':
                rare_table['Recall'].mean(),
            'Training Time (s)': row['Training Time (s)']
        })

linear_comparison_df = pd.DataFrame(linear_records)

assert len(linear_comparison_df) == 8

linear_comparison_df.to_csv(
    TABLES_DIR / 'classical_linear_configuration_comparison.csv',
    index=False
)

print("Linear classifier configuration comparison")

display(linear_comparison_df.round(6))


# Define controlled pairwise comparisons

ablation_pairs = [
    ('Representation', 'Word to Character',
     'Word', 'Character'),

    ('Representation', 'Word to Combined',
     'Word', 'Combined'),

    ('Representation', 'Character to Combined',
     'Character', 'Combined'),

    ('Class Weighting', 'Unweighted to Balanced',
     'Combined', 'Combined Balanced')
]

ablation_records = []
label_ablation_records = []

for model, runs in linear_run_groups.items():

    for experiment, comparison, before, after in ablation_pairs:

        before_id = runs[before]
        after_id = runs[after]

        before_row = overall_lookup.loc[before_id]
        after_row = overall_lookup.loc[after_id]

        before_labels = get_label_metrics(before_id)
        after_labels = get_label_metrics(after_id)

        before_rare = before_labels.loc[rare_labels]
        after_rare = after_labels.loc[rare_labels]

        # Verify that only the intended factor changes

        assert before_row['Model'] == after_row['Model']
        assert before_row['Seed'] == after_row['Seed']
        assert before_row['Features'] == after_row['Features']

        if experiment == 'Representation':

            assert before_row['Class Weight'] == 'None'
            assert after_row['Class Weight'] == 'None'

        else:

            assert before_row['Representation'] == (
                after_row['Representation']
            )

            assert before_row['Class Weight'] == 'None'
            assert after_row['Class Weight'] == 'balanced'

        ap_change = (
            after_labels['Average Precision']
            - before_labels['Average Precision']
        )

        f1_change = (
            after_labels['F1']
            - before_labels['F1']
        )

        ablation_records.append({
            'Model': model,
            'Experiment': experiment,
            'Comparison': comparison,
            'Before Run': before_id,
            'After Run': after_id,
            'Before Macro AP': before_row['Macro AP'],
            'After Macro AP': after_row['Macro AP'],
            'Macro AP Change':
                after_row['Macro AP'] - before_row['Macro AP'],
            'Macro ROC-AUC Change':
                after_row['Macro ROC-AUC']
                - before_row['Macro ROC-AUC'],
            'Default Macro F1 Change':
                after_row['Default Macro F1']
                - before_row['Default Macro F1'],
            'Optimized Macro F1 Change':
                after_row['Optimized Macro F1']
                - before_row['Optimized Macro F1'],
            'Optimized Micro F1 Change':
                after_row['Optimized Micro F1']
                - before_row['Optimized Micro F1'],
            'Rare Label Mean AP Change':
                after_rare['Average Precision'].mean()
                - before_rare['Average Precision'].mean(),
            'Rare Label Mean F1 Change':
                after_rare['F1'].mean()
                - before_rare['F1'].mean(),
            'Rare Label Mean Recall Change':
                after_rare['Recall'].mean()
                - before_rare['Recall'].mean(),
            'Labels with Improved AP':
                int((ap_change > 1e-12).sum()),
            'Labels with Improved F1':
                int((f1_change > 1e-12).sum()),
            'Training Time Ratio':
                after_row['Training Time (s)']
                / before_row['Training Time (s)']
        })

        for label in LABELS:

            old = before_labels.loc[label]
            new = after_labels.loc[label]

            label_ablation_records.append({
                'Model': model,
                'Experiment': experiment,
                'Comparison': comparison,
                'Before Run': before_id,
                'After Run': after_id,
                'Label': label,
                'Before AP': old['Average Precision'],
                'After AP': new['Average Precision'],
                'AP Change':
                    new['Average Precision']
                    - old['Average Precision'],
                'Before F1': old['F1'],
                'After F1': new['F1'],
                'F1 Change': new['F1'] - old['F1'],
                'Precision Change':
                    new['Precision'] - old['Precision'],
                'Recall Change':
                    new['Recall'] - old['Recall'],
                'TP Change': int(new['TP'] - old['TP']),
                'FP Change': int(new['FP'] - old['FP']),
                'FN Change': int(new['FN'] - old['FN'])
            })


ablation_df = pd.DataFrame(ablation_records)

label_ablation_df = pd.DataFrame(
    label_ablation_records
)

assert len(ablation_df) == 8
assert len(label_ablation_df) == 48

assert not label_ablation_df.duplicated([
    'Model', 'Comparison', 'Label'
]).any()

ablation_df.to_csv(
    TABLES_DIR / 'classical_controlled_ablation_comparison.csv',
    index=False
)

label_ablation_df.to_csv(
    TABLES_DIR / 'classical_per_label_ablation_comparison.csv',
    index=False
)


# Analyze feature representation effects

representation_ablation_df = ablation_df.loc[
    ablation_df['Experiment'].eq('Representation')
].copy()

assert len(representation_ablation_df) == 6

print("\nFeature representation ablation")

display(
    representation_ablation_df[[
        'Model',
        'Comparison',
        'Before Macro AP',
        'After Macro AP',
        'Macro AP Change',
        'Optimized Macro F1 Change',
        'Rare Label Mean AP Change',
        'Labels with Improved AP',
        'Training Time Ratio'
    ]].round(6)
)

representation_ablation_df.to_csv(
    TABLES_DIR / 'classical_representation_ablation.csv',
    index=False
)


# Analyze balanced class weighting effects

weighting_ablation_df = ablation_df.loc[
    ablation_df['Experiment'].eq('Class Weighting')
].copy()

assert len(weighting_ablation_df) == 2

print("\nClass weighting ablation")

display(
    weighting_ablation_df[[
        'Model',
        'Before Macro AP',
        'After Macro AP',
        'Macro AP Change',
        'Default Macro F1 Change',
        'Optimized Macro F1 Change',
        'Optimized Micro F1 Change',
        'Rare Label Mean AP Change',
        'Rare Label Mean F1 Change',
        'Rare Label Mean Recall Change',
        'Training Time Ratio'
    ]].round(6)
)

weighting_ablation_df.to_csv(
    TABLES_DIR / 'classical_weighting_ablation.csv',
    index=False
)


# Compare weighting effects on individual labels

weighting_per_label_df = label_ablation_df.loc[
    label_ablation_df['Experiment'].eq('Class Weighting')
].copy()

assert len(weighting_per_label_df) == 12

print("\nPer-label class weighting effects")

display(
    weighting_per_label_df[[
        'Model',
        'Label',
        'AP Change',
        'F1 Change',
        'Precision Change',
        'Recall Change',
        'TP Change',
        'FP Change',
        'FN Change'
    ]].round(6)
)

weighting_per_label_df.to_csv(
    TABLES_DIR / 'classical_weighting_per_label.csv',
    index=False
)


# Examine rare-label changes across all ablations

rare_label_ablation_df = label_ablation_df.loc[
    label_ablation_df['Label'].isin(rare_labels)
].copy()

assert len(rare_label_ablation_df) == 24

rare_label_ablation_df.to_csv(
    TABLES_DIR / 'classical_rare_label_ablation.csv',
    index=False
)


# Select the leading configuration within each linear family

family_selection_records = []

for model in linear_run_groups:

    candidates = (
        linear_comparison_df.loc[
            linear_comparison_df['Model'].eq(model)
        ]
        .sort_values(
            ['Macro AP', 'Optimized Macro F1', 'Run ID'],
            ascending=[False, False, True]
        )
        .reset_index(drop=True)
    )

    selected = candidates.iloc[0]

    family_selection_records.append({
        'Model': model,
        'Selected Run': selected['Run ID'],
        'Configuration': selected['Configuration'],
        'Macro AP': selected['Macro AP'],
        'Optimized Macro F1':
            selected['Optimized Macro F1'],
        'Optimized Micro F1':
            selected['Optimized Micro F1'],
        'Rare Label Mean AP':
            selected['Rare Label Mean AP'],
        'Rare Label Mean F1':
            selected['Rare Label Mean F1'],
        'Selection Metric': 'Macro AP'
    })

linear_family_selection_df = pd.DataFrame(
    family_selection_records
)

print("\nLeading configuration within each linear family")

display(linear_family_selection_df.round(6))

linear_family_selection_df.to_csv(
    TABLES_DIR / 'classical_linear_family_selection.csv',
    index=False
)


# Plot representation performance and weighting effects

fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 6)
)

representation_order = [
    'Word', 'Character', 'Combined'
]

representation_plot_df = (
    linear_comparison_df.loc[
        linear_comparison_df['Configuration'].isin(
            representation_order
        )
    ]
    .pivot(
        index='Configuration',
        columns='Model',
        values='Macro AP'
    )
    .reindex(representation_order)
    .reindex(columns=list(linear_run_groups))
)

representation_plot_df.plot(
    kind='bar',
    ax=axes[0],
    color=[
        MODEL_COLORS[model]
        for model in representation_plot_df.columns
    ],
    width=0.7
)

for container in axes[0].containers:

    axes[0].bar_label(
        container,
        fmt='%.3f',
        padding=3,
        fontsize=10
    )

axes[0].set_title('Effect of Feature Representation')
axes[0].set_ylabel('Validation Macro AP')
axes[0].set_xlabel('')
axes[0].set_ylim(0, 0.85)
axes[0].tick_params(axis='x', rotation=0)
axes[0].grid(axis='x', visible=False)


weighting_plot_df = (
    weighting_ablation_df.set_index('Model')[[
        'Macro AP Change',
        'Default Macro F1 Change',
        'Optimized Macro F1 Change'
    ]]
    .reindex(list(linear_run_groups))
    .T
)

weighting_plot_df.index = [
    'Macro AP',
    'Default Macro F1',
    'Optimized Macro F1'
]

weighting_plot_df *= 100

weighting_plot_df.plot(
    kind='bar',
    ax=axes[1],
    color=[
        MODEL_COLORS[model]
        for model in weighting_plot_df.columns
    ],
    width=0.7
)

for container in axes[1].containers:

    axes[1].bar_label(
        container,
        fmt='%+.2f',
        padding=4,
        fontsize=9
    )

axes[1].axhline(
    0,
    color='black',
    linewidth=0.8
)

axes[1].set_title('Effect of Balanced Class Weighting')
axes[1].set_ylabel('Absolute change (percentage points)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(axis='x', visible=False)
axes[1].margins(y=0.18)

fig.suptitle(
    'Feature Representation and Class Weighting Ablations',
    fontsize=16
)

sns.despine()

fig.tight_layout(
    rect=[0, 0, 1, 0.93]
)

fig.savefig(
    FIGURES_DIR / 'classical_representation_weighting_ablation.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Record ablation findings

ablation_analysis = {
    'models_compared': list(linear_run_groups),
    'configurations_compared': len(linear_comparison_df),
    'representation_comparisons':
        len(representation_ablation_df),
    'weighting_comparisons':
        len(weighting_ablation_df),
    'per_label_comparisons': len(label_ablation_df),
    'rare_labels': rare_labels,
    'feature_budget': FEATURE_BUDGET,
    'training_samples': 59997,
    'validation_samples': len(y_val),
    'seed': SEED,
    'selection_metric': 'macro_average_precision',
    'selected_linear_configurations': {
        row['Model']: row['Selected Run']
        for _, row in linear_family_selection_df.iterrows()
    },
    'class_weighting_tested_on_combined_features_only': True,
    'statistical_significance_established': False,
    'model_training_performed': False,
    'official_test_used': False,
    'final_project_model_frozen': False
}

with open(
    RUN_CONFIGS_DIR / 'classical_ablation_analysis.json',
    'w'
) as file:

    json.dump(
        ablation_analysis,
        file,
        indent=4
    )


# Verify original records remain unchanged

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)


# Summarize the ablation analysis

print("\nClassical ablation analysis summary")

print("Linear configurations compared:",
      len(linear_comparison_df))

print("Representation comparisons:",
      len(representation_ablation_df))

print("Class weighting comparisons:",
      len(weighting_ablation_df))

print("Per-label comparison records:",
      len(label_ablation_df))

print("Rare-label comparison records:",
      len(rare_label_ablation_df))

for _, row in linear_family_selection_df.iterrows():

    print(
        f"\n{row['Model']}: {row['Selected Run']}"
    )

    print("Macro AP:",
          round(row['Macro AP'], 6))

    print("Optimized Macro F1:",
          round(row['Optimized Macro F1'], 6))

for _, row in weighting_ablation_df.iterrows():

    print(
        f"\n{row['Model']} weighting effect"
    )

    print("Macro AP change:",
          round(row['Macro AP Change'], 6))

    print("Optimized Macro F1 change:",
          round(row['Optimized Macro F1 Change'], 6))

    print("Rare-label mean AP change:",
          round(row['Rare Label Mean AP Change'], 6))

print("\nAnalysis tables saved: 8")
print("Research figures generated: 1")
print("Analysis configuration saved: Yes")
print("Original experiment registry: Unchanged")
print("Model training performed: No")
print("Official test dataset: Not loaded")

print("\nClassical ablation analysis completed successfully.")

In [ ]:
# Analyze threshold effectiveness across all configurations

threshold_effectiveness_df = classical_overall_df[[
    'Run ID',
    'Model',
    'Representation',
    'Class Weight',
    'Score Type',
    'Macro AP',
    'Default Macro F1',
    'Optimized Macro F1',
    'Default Micro F1',
    'Optimized Micro F1',
    'Default Macro Precision',
    'Optimized Macro Precision',
    'Default Macro Recall',
    'Optimized Macro Recall',
    'Default Hamming Loss',
    'Optimized Hamming Loss',
    'Default Exact Match Accuracy',
    'Optimized Exact Match Accuracy'
]].copy()

threshold_effectiveness_df['Macro F1 Improvement'] = (
    threshold_effectiveness_df['Optimized Macro F1']
    - threshold_effectiveness_df['Default Macro F1']
)

threshold_effectiveness_df['Micro F1 Improvement'] = (
    threshold_effectiveness_df['Optimized Micro F1']
    - threshold_effectiveness_df['Default Micro F1']
)

threshold_effectiveness_df['Macro Recall Improvement'] = (
    threshold_effectiveness_df['Optimized Macro Recall']
    - threshold_effectiveness_df['Default Macro Recall']
)

threshold_effectiveness_df['Macro Precision Change'] = (
    threshold_effectiveness_df['Optimized Macro Precision']
    - threshold_effectiveness_df['Default Macro Precision']
)

threshold_effectiveness_df['Hamming Loss Reduction'] = (
    threshold_effectiveness_df['Default Hamming Loss']
    - threshold_effectiveness_df['Optimized Hamming Loss']
)

threshold_effectiveness_df['Exact Match Accuracy Change'] = (
    threshold_effectiveness_df['Optimized Exact Match Accuracy']
    - threshold_effectiveness_df['Default Exact Match Accuracy']
)

assert len(threshold_effectiveness_df) == 11

assert (
    threshold_effectiveness_df['Macro F1 Improvement']
    >= -1e-12
).all()

threshold_effectiveness_df = (
    threshold_effectiveness_df.sort_values(
        'Macro F1 Improvement',
        ascending=False
    )
    .reset_index(drop=True)
)

threshold_effectiveness_df.to_csv(
    TABLES_DIR / 'classical_threshold_effectiveness.csv',
    index=False
)

print("Threshold effectiveness across all configurations")

display(
    threshold_effectiveness_df[[
        'Run ID',
        'Default Macro F1',
        'Optimized Macro F1',
        'Macro F1 Improvement',
        'Default Micro F1',
        'Optimized Micro F1',
        'Micro F1 Improvement'
    ]].round(6)
)


# Analyze threshold optimization at the label level

threshold_label_analysis_df = classical_threshold_df.copy()

assert len(threshold_label_analysis_df) == 66

threshold_label_analysis_df['F1 Improved'] = (
    threshold_label_analysis_df['F1 Improvement'] > 1e-12
)

threshold_label_analysis_df['Zero Default Predictions'] = (
    threshold_label_analysis_df[
        'Default Predicted Positives'
    ] == 0
)

threshold_label_analysis_df['Positive Predictions Recovered'] = (
    threshold_label_analysis_df['Zero Default Predictions']
    & (
        threshold_label_analysis_df[
            'Optimized Predicted Positives'
        ] > 0
    )
)

threshold_label_analysis_df.to_csv(
    TABLES_DIR / 'classical_threshold_label_effectiveness.csv',
    index=False
)


# Summarize threshold sensitivity for each toxicity label

label_threshold_summary_df = (
    threshold_label_analysis_df.groupby('Label')
    .agg(
        Configurations=('Run ID', 'count'),
        Mean_Default_F1=('Default F1', 'mean'),
        Mean_Optimized_F1=('Optimized F1', 'mean'),
        Mean_F1_Improvement=('F1 Improvement', 'mean'),
        Maximum_F1_Improvement=('F1 Improvement', 'max'),
        Configurations_Improved=('F1 Improved', 'sum'),
        Zero_Default_Predictions=(
            'Zero Default Predictions', 'sum'
        ),
        Positive_Predictions_Recovered=(
            'Positive Predictions Recovered', 'sum'
        )
    )
    .reindex(LABELS)
    .reset_index()
)

assert label_threshold_summary_df[
    'Configurations'
].eq(11).all()

print("\nLabel-wise threshold effectiveness")

display(label_threshold_summary_df.round(6))

label_threshold_summary_df.to_csv(
    TABLES_DIR / 'classical_threshold_label_summary.csv',
    index=False
)


# Compare rare and common label threshold improvements

threshold_label_analysis_df['Label Group'] = np.where(
    threshold_label_analysis_df['Label'].isin(rare_labels),
    'Rare',
    'Common'
)

threshold_group_summary_df = (
    threshold_label_analysis_df
    .groupby(['Run ID', 'Model', 'Label Group'])
    .agg(
        Mean_Default_F1=('Default F1', 'mean'),
        Mean_Optimized_F1=('Optimized F1', 'mean'),
        Mean_F1_Improvement=('F1 Improvement', 'mean'),
        Labels_Improved=('F1 Improved', 'sum')
    )
    .reset_index()
)

assert len(threshold_group_summary_df) == 22

threshold_group_summary_df.to_csv(
    TABLES_DIR / 'classical_threshold_group_comparison.csv',
    index=False
)

print("\nRare and common label threshold improvements")

display(
    threshold_group_summary_df.round(6)
)


# Consolidate recorded computational measurements

efficiency_df = classical_computational_df.merge(
    classical_overall_df[[
        'Run ID',
        'Macro AP',
        'Macro ROC-AUC',
        'Optimized Macro F1',
        'Optimized Micro F1'
    ]],
    on='Run ID',
    validate='one_to_one'
)

assert len(efficiency_df) == 11
assert efficiency_df['Training Time (s)'].gt(0).all()

assert efficiency_df[
    'Validation Prediction Time (s)'
].notna().all()

assert efficiency_df[
    'Validation Prediction Time (s)'
].gt(0).all()

assert efficiency_df['Model Size (MiB)'].gt(0).all()

efficiency_df['Validation Comments per Second'] = (
    len(y_val)
    / efficiency_df['Validation Prediction Time (s)']
)

efficiency_df['Prediction Time per 1000 Comments (s)'] = (
    1000
    * efficiency_df['Validation Prediction Time (s)']
    / len(y_val)
)

efficiency_df['Training and Prediction Time (s)'] = (
    efficiency_df['Training Time (s)']
    + efficiency_df['Validation Prediction Time (s)']
)


# Identify configurations on the training-time Pareto frontier

pareto_records = []

for _, candidate in efficiency_df.iterrows():

    dominates = (
        (
            efficiency_df['Macro AP']
            >= candidate['Macro AP']
        )
        & (
            efficiency_df['Training Time (s)']
            <= candidate['Training Time (s)']
        )
        & (
            (
                efficiency_df['Macro AP']
                > candidate['Macro AP']
            )
            | (
                efficiency_df['Training Time (s)']
                < candidate['Training Time (s)']
            )
        )
    )

    dominating_runs = efficiency_df.loc[
        dominates, 'Run ID'
    ].tolist()

    pareto_records.append({
        'Run ID': candidate['Run ID'],
        'Pareto Efficient': len(dominating_runs) == 0,
        'Dominating Configurations': len(dominating_runs),
        'Dominated By': ', '.join(dominating_runs)
    })

pareto_df = pd.DataFrame(pareto_records)

efficiency_df = efficiency_df.merge(
    pareto_df,
    on='Run ID',
    validate='one_to_one'
)

efficiency_df = (
    efficiency_df.sort_values(
        'Training Time (s)'
    )
    .reset_index(drop=True)
)

efficient_configurations_df = efficiency_df.loc[
    efficiency_df['Pareto Efficient']
].copy()

assert not efficient_configurations_df.empty

efficiency_df.to_csv(
    TABLES_DIR / 'classical_computational_efficiency.csv',
    index=False
)

efficient_configurations_df.to_csv(
    TABLES_DIR / 'classical_pareto_efficient_configurations.csv',
    index=False
)

print("\nComputational efficiency comparison")

display(
    efficiency_df[[
        'Run ID',
        'Macro AP',
        'Optimized Macro F1',
        'Training Time (s)',
        'Validation Prediction Time (s)',
        'Validation Comments per Second',
        'Model Size (MiB)',
        'Pareto Efficient'
    ]].round(6)
)

print("\nTraining-time Pareto-efficient configurations")

display(
    efficient_configurations_df[[
        'Run ID',
        'Macro AP',
        'Training Time (s)',
        'Optimized Macro F1'
    ]].round(6)
)


# Prepare readable configuration names

short_run_names = {
    'NB_WORD_BASELINE_S42': 'NB Word',
    'LR_WORD_BASELINE_S42': 'LR Word',
    'LR_CHAR_BASELINE_S42': 'LR Char',
    'LR_COMBINED_BASELINE_S42': 'LR Combined',
    'LR_COMBINED_BALANCED_S42': 'LR Balanced',
    'SVM_WORD_BASELINE_S42': 'SVM Word',
    'SVM_CHAR_BASELINE_S42': 'SVM Char',
    'SVM_COMBINED_BASELINE_S42': 'SVM Combined',
    'SVM_COMBINED_BALANCED_S42': 'SVM Balanced',
    'DT_WORD_BASELINE_S42': 'DT Word',
    'RF_WORD_BASELINE_S42': 'RF Word'
}

assert set(short_run_names) == set(expected_run_ids)


# Plot threshold effectiveness and computational trade-offs

fig, axes = plt.subplots(
    1, 2,
    figsize=(17, 8)
)

threshold_plot_df = threshold_effectiveness_df.copy()

threshold_plot_df['Configuration'] = (
    threshold_plot_df['Run ID'].map(short_run_names)
)

threshold_plot_df = threshold_plot_df.set_index(
    'Configuration'
)

threshold_plot_df[[
    'Default Macro F1',
    'Optimized Macro F1'
]].plot(
    kind='barh',
    ax=axes[0],
    color=['#94A3B8', '#4C78A8'],
    width=0.75
)

axes[0].invert_yaxis()
axes[0].set_title('Effect of Per-label Threshold Optimization')
axes[0].set_xlabel('Validation Macro F1')
axes[0].set_ylabel('')
axes[0].set_xlim(0, 0.85)
axes[0].grid(axis='y', visible=False)
axes[0].legend(['Default', 'Optimized'])

for container in axes[0].containers:

    axes[0].bar_label(
        container,
        fmt='%.3f',
        padding=3,
        fontsize=8
    )


# Plot performance against recorded training time

for model, group in efficiency_df.groupby('Model'):

    axes[1].scatter(
        group['Training Time (s)'],
        group['Macro AP'],
        color=MODEL_COLORS[model],
        s=85,
        edgecolors='white',
        linewidths=0.8,
        label=short_names[model],
        zorder=3
    )

frontier = efficient_configurations_df.sort_values(
    'Training Time (s)'
)

axes[1].plot(
    frontier['Training Time (s)'],
    frontier['Macro AP'],
    color='gray',
    linestyle='--',
    linewidth=1.3,
    label='Pareto frontier',
    zorder=2
)

annotation_offsets = {
    'NB_WORD_BASELINE_S42': (6, 5),
    'LR_WORD_BASELINE_S42': (-8, -15),
    'SVM_WORD_BASELINE_S42': (-8, 8),
    'LR_CHAR_BASELINE_S42': (-12, -18),
    'LR_COMBINED_BASELINE_S42': (10, -8),
    'SVM_CHAR_BASELINE_S42': (-12, 8),
    'SVM_COMBINED_BASELINE_S42': (8, 14),
    'LR_COMBINED_BALANCED_S42': (6, 8),
    'SVM_COMBINED_BALANCED_S42': (6, -15),
    'RF_WORD_BASELINE_S42': (6, -12),
    'DT_WORD_BASELINE_S42': (-55, 8)
}

for _, row in efficiency_df.iterrows():

    axes[1].annotate(
        short_run_names[row['Run ID']],
        (
            row['Training Time (s)'],
            row['Macro AP']
        ),
        xytext=annotation_offsets[row['Run ID']],
        textcoords='offset points',
        fontsize=8
    )

axes[1].set_xscale('log')
axes[1].set_xlim(0.4, 400)
axes[1].set_ylim(0.35, 0.76)
axes[1].set_title('Performance versus Recorded Training Time')
axes[1].set_xlabel('Training time (seconds, log scale)')
axes[1].set_ylabel('Validation Macro AP')
axes[1].legend(loc='lower center', fontsize=8, ncol=2)

fig.suptitle(
    'Classical ML Threshold and Computational Analysis',
    fontsize=16
)

sns.despine()

fig.tight_layout(
    rect=[0, 0, 1, 0.95]
)

fig.savefig(
    FIGURES_DIR / 'classical_threshold_efficiency_analysis.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Identify the strongest threshold improvement and fastest runs

largest_threshold_gain = threshold_effectiveness_df.iloc[0]

fastest_training = efficiency_df.loc[
    efficiency_df['Training Time (s)'].idxmin()
]

fastest_prediction = efficiency_df.loc[
    efficiency_df['Validation Prediction Time (s)'].idxmin()
]

smallest_model = efficiency_df.loc[
    efficiency_df['Model Size (MiB)'].idxmin()
]

leading_configuration = efficiency_df.loc[
    efficiency_df['Macro AP'].idxmax()
]


# Record findings and measurement limitations

threshold_efficiency_analysis = {
    'configurations_compared': len(efficiency_df),
    'label_threshold_comparisons':
        len(threshold_label_analysis_df),
    'configurations_with_improved_macro_f1': int(
        (
            threshold_effectiveness_df['Macro F1 Improvement']
            > 1e-12
        ).sum()
    ),
    'individual_label_f1_improvements': int(
        threshold_label_analysis_df['F1 Improved'].sum()
    ),
    'zero_default_prediction_cases': int(
        threshold_label_analysis_df[
            'Zero Default Predictions'
        ].sum()
    ),
    'recovered_positive_prediction_cases': int(
        threshold_label_analysis_df[
            'Positive Predictions Recovered'
        ].sum()
    ),
    'largest_threshold_improvement_run':
        str(largest_threshold_gain['Run ID']),
    'largest_macro_f1_improvement': float(
        largest_threshold_gain['Macro F1 Improvement']
    ),
    'fastest_training_run':
        str(fastest_training['Run ID']),
    'fastest_prediction_run':
        str(fastest_prediction['Run ID']),
    'smallest_saved_model_run':
        str(smallest_model['Run ID']),
    'highest_macro_ap_run':
        str(leading_configuration['Run ID']),
    'pareto_efficient_runs':
        efficient_configurations_df['Run ID'].tolist(),
    'pareto_objectives': {
        'maximize': 'validation_macro_ap',
        'minimize': 'recorded_training_time'
    },
    'runtime_comparison_descriptive_only': True,
    'runtime_repetitions_performed': False,
    'identical_parallelization': False,
    'feature_extraction_time_included': False,
    'vectorizer_size_included': False,
    'prediction_time_includes_raw_text_preprocessing': False,
    'thresholds_selected_on_validation': True,
    'probability_calibration_established': False,
    'statistical_significance_established': False,
    'model_training_performed': False,
    'official_test_used': False,
    'final_project_model_frozen': False
}

with open(
    RUN_CONFIGS_DIR
    / 'classical_threshold_efficiency_analysis.json',
    'w'
) as file:

    json.dump(
        threshold_efficiency_analysis,
        file,
        indent=4
    )


# Verify source records remain unchanged

assert calculate_sha256(REGISTRY_PATH) == registry_original_hash

assert calculate_sha256(
    DECISION_LOG_PATH
) == decision_log_original_hash


# Summarize threshold and efficiency analysis

print("\nClassical threshold and efficiency analysis summary")

print("Configurations compared:", len(efficiency_df))

print(
    "Configurations with improved Macro F1:",
    threshold_efficiency_analysis[
        'configurations_with_improved_macro_f1'
    ],
    "/ 11"
)

print(
    "Individual label F1 improvements:",
    threshold_efficiency_analysis[
        'individual_label_f1_improvements'
    ],
    "/ 66"
)

print(
    "Zero default prediction cases:",
    threshold_efficiency_analysis[
        'zero_default_prediction_cases'
    ]
)

print(
    "Recovered positive prediction cases:",
    threshold_efficiency_analysis[
        'recovered_positive_prediction_cases'
    ]
)

print(
    "Largest threshold improvement:",
    largest_threshold_gain['Run ID']
)

print(
    "Maximum Macro F1 improvement:",
    round(
        largest_threshold_gain['Macro F1 Improvement'],
        6
    )
)

print("\nFastest training:", fastest_training['Run ID'])

print(
    "Training time (s):",
    round(fastest_training['Training Time (s)'], 6)
)

print("Fastest prediction:", fastest_prediction['Run ID'])

print(
    "Validation prediction time (s):",
    round(
        fastest_prediction['Validation Prediction Time (s)'],
        6
    )
)

print("Smallest saved model:", smallest_model['Run ID'])

print(
    "Model size (MiB):",
    round(smallest_model['Model Size (MiB)'], 6)
)

print(
    "\nPareto-efficient configurations:",
    len(efficient_configurations_df)
)

print(
    efficient_configurations_df['Run ID'].tolist()
)

print("\nAnalysis tables saved: 6")
print("Research figures generated: 1")
print("Analysis configuration saved: Yes")
print("Original experiment registry: Unchanged")
print("Model training performed: No")
print("Official test dataset: Not loaded")

print(
    "\nClassical threshold and efficiency analysis "
    "completed successfully."
)

In [ ]:
# Select the leading classical configurations

assert len(classical_results) == 11
assert len(classical_ranking_df) == 11

primary_run_id = str(
    classical_ranking_df.iloc[0]['Run ID']
)

secondary_run_id = str(
    classical_ranking_df.loc[
        classical_ranking_df['Model'].eq('Linear SVM')
    ].iloc[0]['Run ID']
)

assert primary_run_id != secondary_run_id

selected_runs = {
    'Primary Classical Baseline': primary_run_id,
    'Secondary Classical Baseline': secondary_run_id
}

overall_lookup = classical_overall_df.set_index('Run ID')

selection_records = []

for role, run_id in selected_runs.items():

    row = overall_lookup.loc[run_id]

    selection_records.append({
        'Selection Role': role,
        'Run ID': run_id,
        'Model': row['Model'],
        'Representation': row['Representation'],
        'Class Weight': row['Class Weight'],
        'Macro AP': row['Macro AP'],
        'Macro ROC-AUC': row['Macro ROC-AUC'],
        'Default Macro F1': row['Default Macro F1'],
        'Optimized Macro F1': row['Optimized Macro F1'],
        'Optimized Micro F1': row['Optimized Micro F1'],
        'Training Time (s)': row['Training Time (s)'],
        'Validation Prediction Time (s)':
            row['Validation Prediction Time (s)'],
        'Model Size (MiB)': row['Model Size (MiB)']
    })

classical_selection_df = pd.DataFrame(selection_records)

classical_selection_df.to_csv(
    TABLES_DIR / 'classical_selected_baselines.csv',
    index=False
)

print("Selected classical baselines")

display(classical_selection_df.round(6))


# Calculate performance differences between selected models

primary_row = overall_lookup.loc[primary_run_id]
secondary_row = overall_lookup.loc[secondary_run_id]

selected_comparison_metrics = [
    'Macro AP',
    'Macro ROC-AUC',
    'Default Macro F1',
    'Optimized Macro F1',
    'Optimized Micro F1',
    'Optimized Macro Precision',
    'Optimized Macro Recall',
    'Optimized Hamming Loss',
    'Optimized Exact Match Accuracy',
    'Training Time (s)',
    'Validation Prediction Time (s)',
    'Model Size (MiB)'
]

selected_comparison_df = pd.DataFrame([
    {
        'Metric': metric,
        'Primary Baseline': primary_row[metric],
        'Secondary Baseline': secondary_row[metric],
        'Primary minus Secondary':
            primary_row[metric] - secondary_row[metric]
    }
    for metric in selected_comparison_metrics
])

selected_comparison_df.to_csv(
    TABLES_DIR / 'classical_selected_baseline_comparison.csv',
    index=False
)

print("\nSelected baseline comparison")

display(selected_comparison_df.round(6))


# Consolidate selected per-label metrics

selected_per_label_df = classical_per_label_df.loc[
    classical_per_label_df['Run ID'].isin(selected_runs.values())
    & classical_per_label_df['Threshold Type'].eq('Optimized')
].copy()

assert len(selected_per_label_df) == 12

selected_per_label_df.to_csv(
    TABLES_DIR / 'classical_selected_per_label_metrics.csv',
    index=False
)

selected_ap_df = (
    selected_per_label_df.pivot(
        index='Label',
        columns='Run ID',
        values='Average Precision'
    )
    .reindex(index=LABELS)
    .reindex(columns=list(selected_runs.values()))
)

selected_f1_df = (
    selected_per_label_df.pivot(
        index='Label',
        columns='Run ID',
        values='F1'
    )
    .reindex(index=LABELS)
    .reindex(columns=list(selected_runs.values()))
)

print("\nSelected models: per-label AP")

display(selected_ap_df.round(6))

print("\nSelected models: per-label optimized F1")

display(selected_f1_df.round(6))

selected_ap_df.to_csv(
    TABLES_DIR / 'classical_selected_ap_matrix.csv'
)

selected_f1_df.to_csv(
    TABLES_DIR / 'classical_selected_f1_matrix.csv'
)


# Preserve the exact validation-selected thresholds

selected_thresholds_df = classical_threshold_df.loc[
    classical_threshold_df['Run ID'].isin(selected_runs.values())
].copy()

assert len(selected_thresholds_df) == 12

selected_thresholds_df.to_csv(
    TABLES_DIR / 'classical_selected_thresholds.csv',
    index=False
)

print("\nSelected baseline thresholds")

display(
    selected_thresholds_df[[
        'Run ID',
        'Label',
        'Score Type',
        'Default Threshold',
        'Optimized Threshold',
        'Optimized F1'
    ]].round(6)
)


# Verify shared feature dependencies

feature_manifest = pd.read_csv(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
)

assert calculate_sha256(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
) == feature_manifest_hash

vectorizer_dir = (
    PROJECT_ROOT / '04_models' / 'classical' / 'vectorizers'
)

feature_dependency_paths = {
    'combined_word_vectorizer':
        vectorizer_dir / 'combined_word_tfidf.joblib',

    'combined_character_vectorizer':
        vectorizer_dir / 'combined_character_tfidf.joblib'
}

dependency_records = []

for dependency, path in feature_dependency_paths.items():

    assert path.is_file()

    relative_path = str(path.relative_to(PROJECT_ROOT))

    manifest_record = feature_manifest.loc[
        feature_manifest['Path'].eq(relative_path)
        & feature_manifest['Artifact Type'].eq('Vectorizer')
    ]

    assert len(manifest_record) == 1

    actual_hash = calculate_sha256(path)

    assert actual_hash == manifest_record.iloc[0]['SHA256']

    dependency_records.append({
        'Dependency': dependency,
        'Path': relative_path,
        'Size Bytes': path.stat().st_size,
        'SHA256': actual_hash,
        'Verification': 'Passed'
    })

selected_feature_dependencies_df = pd.DataFrame(
    dependency_records
)

selected_feature_dependencies_df.to_csv(
    TABLES_DIR / 'classical_selected_feature_dependencies.csv',
    index=False
)

print("\nSelected feature dependencies")

display(selected_feature_dependencies_df)


# Calculate precision-recall curves for both selected models

pr_curve_tables = []

selected_display_names = {
    primary_run_id: 'Balanced Combined Logistic Regression',
    secondary_run_id: 'Unweighted Combined Linear SVM'
}

selected_colors = {
    primary_run_id: MODEL_COLORS['Logistic Regression'],
    secondary_run_id: MODEL_COLORS['Linear SVM']
}

fig, axes = plt.subplots(
    2, 3,
    figsize=(17, 10)
)

for i, label in enumerate(LABELS):

    ax = axes.flat[i]
    targets = y_val[:, i]

    prevalence = float(targets.mean())

    for run_id in selected_runs.values():

        scores = classical_results[run_id]['scores'][:, i]

        precision, recall, curve_thresholds = (
            precision_recall_curve(targets, scores)
        )

        assert len(precision) == len(recall)
        assert len(precision) == len(curve_thresholds) + 1

        average_precision = float(
            selected_ap_df.loc[label, run_id]
        )

        ax.step(
            recall,
            precision,
            where='post',
            linewidth=1.8,
            color=selected_colors[run_id],
            label=(
                f"{selected_display_names[run_id]}"
                f"\nAP = {average_precision:.3f}"
            )
        )

        pr_curve_tables.append(pd.DataFrame({
            'Run ID': run_id,
            'Label': label,
            'Recall': recall,
            'Precision': precision,
            'Score Threshold': np.append(
                curve_thresholds, np.nan
            ),
            'Average Precision': average_precision,
            'Positive Prevalence': prevalence
        }))

    ax.axhline(
        prevalence,
        color='gray',
        linestyle='--',
        linewidth=1.2,
        label=f'Prevalence = {prevalence:.3f}'
    )

    ax.set_title(
        label.replace('_', ' ').title()
    )

    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.03)
    ax.legend(loc='upper right', fontsize=8)

fig.suptitle(
    'Precision-Recall Curves of Selected Classical Baselines',
    fontsize=16
)

sns.despine()

fig.tight_layout(rect=[0, 0, 1, 0.95])

fig.savefig(
    FIGURES_DIR / 'classical_selected_precision_recall_curves.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()
plt.close(fig)


# Save numerical source data for precision-recall curves

selected_pr_curves_df = pd.concat(
    pr_curve_tables,
    ignore_index=True
)

selected_pr_curves_df.to_csv(
    TABLES_DIR / 'classical_selected_pr_curves.csv.gz',
    index=False,
    compression='gzip'
)

print("\nPrecision-recall curve records:",
      len(selected_pr_curves_df))


# Build the consolidated classical selection configuration

selected_model_configs = {}

for role, run_id in selected_runs.items():

    artifacts = run_artifacts[run_id]
    result = classical_results[run_id]
    row = overall_lookup.loc[run_id]

    assert result['representation'] == 'combined_tfidf'

    for path_key, hash_key in [
        ('model_path', 'model_sha256'),
        ('config_path', 'config_sha256'),
        ('threshold_path', 'threshold_sha256'),
        ('predictions_path', 'prediction_sha256')
    ]:

        assert calculate_sha256(
            artifacts[path_key]
        ) == artifacts[hash_key]

    selected_model_configs[role] = {
        'run_id': run_id,
        'model': result['model'],
        'representation': result['representation'],
        'class_weight': result['class_weight'],
        'score_type': result['score_type'],
        'training_samples': int(row['Training Samples']),
        'validation_samples': int(row['Validation Samples']),
        'feature_budget': FEATURE_BUDGET,
        'seed': SEED,
        'default_threshold': float(result['default_threshold']),
        'optimized_thresholds': {
            label: float(value)
            for label, value in zip(
                LABELS,
                result['optimized_thresholds']
            )
        },
        'threshold_comparison_operator': '>=',
        'threshold_selection_partition': 'validation',
        'validation_macro_ap': float(row['Macro AP']),
        'validation_macro_roc_auc': float(row['Macro ROC-AUC']),
        'validation_default_macro_f1':
            float(row['Default Macro F1']),
        'validation_optimized_macro_f1':
            float(row['Optimized Macro F1']),
        'validation_optimized_micro_f1':
            float(row['Optimized Micro F1']),
        'model_path': str(
            artifacts['model_path'].relative_to(PROJECT_ROOT)
        ),
        'model_sha256': artifacts['model_sha256'],
        'config_path': str(
            artifacts['config_path'].relative_to(PROJECT_ROOT)
        ),
        'config_sha256': artifacts['config_sha256'],
        'threshold_path': str(
            artifacts['threshold_path'].relative_to(PROJECT_ROOT)
        ),
        'threshold_sha256': artifacts['threshold_sha256'],
        'predictions_path': str(
            artifacts['predictions_path'].relative_to(PROJECT_ROOT)
        ),
        'predictions_sha256': artifacts['prediction_sha256'],
        'official_test_used': False
    }


classical_selection = {
    'selection_stage': 'representative_subset_classical_comparison',
    'selection_partition': 'validation',
    'primary_selection_metric': 'macro_average_precision',
    'secondary_selection_metric': 'optimized_macro_f1',
    'configurations_compared': len(classical_results),
    'labels': LABELS,
    'seed': SEED,
    'primary_selected_run_id': primary_run_id,
    'secondary_selected_run_id': secondary_run_id,
    'selected_models': selected_model_configs,
    'feature_dependencies': dependency_records,
    'combined_feature_construction': {
        'word_features': 25000,
        'character_features': 25000,
        'concatenation_order': ['word', 'character'],
        'final_normalization': 'l2'
    },
    'training_dataset_sha256': EXPECTED_TRAINING_HASH,
    'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,
    'evaluation_module_sha256': evaluation_hash,
    'feature_manifest_sha256': feature_manifest_hash,
    'full_data_scaling_candidate': primary_run_id,
    'full_data_scaling_completed': False,
    'additional_classical_tuning_planned': False,
    'statistical_significance_established': False,
    'probability_calibration_established': False,
    'model_training_performed': False,
    'official_test_used': False,
    'final_project_model_frozen': False
}

CLASSICAL_SELECTION_PATH = (
    RUN_CONFIGS_DIR / 'classical_final_selection.json'
)

with open(CLASSICAL_SELECTION_PATH, 'w') as file:

    json.dump(
        classical_selection,
        file,
        indent=4
    )

with open(CLASSICAL_SELECTION_PATH, 'r') as file:

    saved_classical_selection = json.load(file)

assert saved_classical_selection == classical_selection


# Verify saved thresholds and source artifacts

for role, selected_config in (
    saved_classical_selection['selected_models'].items()
):

    run_id = selected_config['run_id']

    restored_thresholds = np.array([
        selected_config['optimized_thresholds'][label]
        for label in LABELS
    ])

    result = classical_results[run_id]

    np.testing.assert_array_equal(
        restored_thresholds,
        result['optimized_thresholds']
    )

    assert np.array_equal(
        apply_thresholds(
            result['scores'],
            restored_thresholds
        ),
        result['optimized_predictions']
    )

assert calculate_sha256(REGISTRY_PATH) == registry_original_hash

assert calculate_sha256(
    DECISION_LOG_PATH
) == decision_log_original_hash


# Summarize the completed classical selection

print("\nFinal classical baseline selection summary")

print("Configurations evaluated:", len(classical_results))

print("\nPrimary selected configuration:", primary_run_id)

print("Macro AP:",
      round(primary_row['Macro AP'], 6))

print("Macro ROC-AUC:",
      round(primary_row['Macro ROC-AUC'], 6))

print("Optimized Macro F1:",
      round(primary_row['Optimized Macro F1'], 6))

print("Optimized Micro F1:",
      round(primary_row['Optimized Micro F1'], 6))

print("\nSecondary selected configuration:", secondary_run_id)

print("Macro AP:",
      round(secondary_row['Macro AP'], 6))

print("Optimized Macro F1:",
      round(secondary_row['Optimized Macro F1'], 6))

print("Optimized Micro F1:",
      round(secondary_row['Optimized Micro F1'], 6))

print("\nMacro AP difference:",
      round(
          primary_row['Macro AP']
          - secondary_row['Macro AP'],
          6
      ))

print("Selected baseline threshold records:",
      len(selected_thresholds_df))

print("Feature dependencies verified:",
      len(selected_feature_dependencies_df))

print("Precision-recall curves generated:", 12)
print("Research figures generated: 1")
print("Analysis tables saved: 8")

print("Selection configuration saved:",
      CLASSICAL_SELECTION_PATH.name)

print("Selection configuration roundtrip: Passed")
print("Selected threshold reconstruction: Passed")
print("Original experiment artifacts: Unchanged")
print("Full-data scaling candidate:", primary_run_id)
print("Full-data scaling completed: No")
print("Additional model training: No")
print("Official test dataset: Not loaded")
print("Final project model frozen: No")

gc.collect()

print("\nClassical baseline selection completed successfully.")

In [ ]:
# Verify completion of all classical comparison stages

assert len(classical_results) == 11
assert len(classical_overall_df) == 11
assert len(classical_per_label_df) == 132
assert len(classical_threshold_df) == 66
assert len(classical_reconstruction_df) == 11

assert classical_reconstruction_df['Verification'].eq('Passed').all()

for column in [
    'Maximum Threshold Difference',
    'Macro AP Reconstruction Error',
    'Macro F1 Reconstruction Error'
]:
    assert classical_reconstruction_df[column].abs().max() <= 1e-12

assert not classical_overall_df['Official Test Used'].any()

assert len(common_baseline_df) == 5
assert len(linear_comparison_df) == 8
assert len(ablation_df) == 8
assert len(label_ablation_df) == 48
assert len(threshold_effectiveness_df) == 11
assert len(efficient_configurations_df) == 6

print("All classical analysis stages: Verified")


# Verify the saved classical selection

with open(CLASSICAL_SELECTION_PATH, 'r') as file:
    saved_selection = json.load(file)

assert saved_selection == classical_selection
assert saved_selection['primary_selected_run_id'] == primary_run_id
assert saved_selection['secondary_selected_run_id'] == secondary_run_id
assert saved_selection['selection_partition'] == 'validation'
assert not saved_selection['official_test_used']
assert not saved_selection['final_project_model_frozen']

assert primary_run_id == classical_ranking_df.iloc[0]['Run ID']

for role, selected_config in saved_selection['selected_models'].items():

    run_id = selected_config['run_id']
    result = classical_results[run_id]

    thresholds = np.array([
        selected_config['optimized_thresholds'][label]
        for label in LABELS
    ])

    np.testing.assert_array_equal(
        thresholds,
        result['optimized_thresholds']
    )

    assert np.array_equal(
        apply_thresholds(result['scores'], thresholds),
        result['optimized_predictions']
    )

print("Classical model selection: Verified")
print("Selected thresholds: Verified")


# Verify original experiment artifacts

for _, artifact in source_artifact_df.iterrows():

    path = PROJECT_ROOT / artifact['Path']

    assert path.is_file()
    assert path.stat().st_size == artifact['Size Bytes']
    assert calculate_sha256(path) == artifact['SHA256']

assert len(source_artifact_df) == 44

assert calculate_sha256(REGISTRY_PATH) == registry_original_hash

assert calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
) == evaluation_hash

assert calculate_sha256(
    FEATURES_DIR / 'feature_artifact_manifest.csv'
) == feature_manifest_hash

for dependency in saved_selection['feature_dependencies']:

    path = PROJECT_ROOT / dependency['Path']

    assert path.is_file()
    assert calculate_sha256(path) == dependency['SHA256']

print("Original source fingerprints verified:", len(source_artifact_df))
print("Selected feature dependencies verified:", len(
    saved_selection['feature_dependencies']
))


# Define all required consolidated tables

required_tables = [

    # Cell 1
    'classical_experiment_verification.csv',
    'classical_threshold_format_verification.csv',
    'classical_dataset_consistency.csv',
    'classical_experiment_coverage.csv',
    'environment_versions.csv',
    'environment_compatibility.csv',
    'classical_source_artifact_manifest.csv',

    # Cell 2
    'classical_validation_label_support.csv',
    'classical_all_run_metrics.csv',
    'classical_all_per_label_metrics.csv',
    'classical_all_thresholds.csv',
    'classical_metric_reconstruction.csv',
    'classical_computational_measurements.csv',
    'classical_validation_ranking.csv',

    # Cell 3
    'classical_common_algorithm_comparison.csv',
    'classical_common_algorithm_per_label.csv',
    'classical_common_algorithm_ap_matrix.csv',
    'classical_common_algorithm_f1_matrix.csv',
    'classical_common_algorithm_label_winners.csv',
    'classical_common_algorithm_rare_labels.csv',

    # Cell 4
    'classical_linear_configuration_comparison.csv',
    'classical_controlled_ablation_comparison.csv',
    'classical_per_label_ablation_comparison.csv',
    'classical_representation_ablation.csv',
    'classical_weighting_ablation.csv',
    'classical_weighting_per_label.csv',
    'classical_rare_label_ablation.csv',
    'classical_linear_family_selection.csv',

    # Cell 5
    'classical_threshold_effectiveness.csv',
    'classical_threshold_label_effectiveness.csv',
    'classical_threshold_label_summary.csv',
    'classical_threshold_group_comparison.csv',
    'classical_computational_efficiency.csv',
    'classical_pareto_efficient_configurations.csv',

    # Cell 6
    'classical_selected_baselines.csv',
    'classical_selected_baseline_comparison.csv',
    'classical_selected_per_label_metrics.csv',
    'classical_selected_ap_matrix.csv',
    'classical_selected_f1_matrix.csv',
    'classical_selected_thresholds.csv',
    'classical_selected_feature_dependencies.csv',
    'classical_selected_pr_curves.csv.gz'
]

required_figures = [
    'classical_common_algorithm_comparison.png',
    'classical_common_algorithm_per_label_heatmaps.png',
    'classical_representation_weighting_ablation.png',
    'classical_threshold_efficiency_analysis.png',
    'classical_selected_precision_recall_curves.png'
]

required_configs = [
    'classical_comparison_config.json',
    'classical_common_algorithm_analysis.json',
    'classical_ablation_analysis.json',
    'classical_threshold_efficiency_analysis.json',
    'classical_final_selection.json'
]

required_logs = [
    'classical_reconstruction_summary.json'
]


# Verify consolidated artifact completeness

verification_records = []

for folder, filenames, category in [
    (TABLES_DIR, required_tables, 'Analysis Table'),
    (FIGURES_DIR, required_figures, 'Figure'),
    (RUN_CONFIGS_DIR, required_configs, 'Configuration'),
    (LOGS_DIR, required_logs, 'Log')
]:

    for filename in filenames:

        path = folder / filename

        available = path.is_file() and path.stat().st_size > 0

        verification_records.append({
            'Type': category,
            'Filename': filename,
            'Path': str(path.relative_to(PROJECT_ROOT)),
            'Size Bytes': path.stat().st_size if available else 0,
            'SHA256': calculate_sha256(path) if available else '',
            'Status': 'Available' if available else 'Missing'
        })

final_verification_df = pd.DataFrame(verification_records)

assert final_verification_df['Path'].is_unique

missing_artifacts_df = final_verification_df.loc[
    final_verification_df['Status'].eq('Missing')
]

if not missing_artifacts_df.empty:
    display(missing_artifacts_df)

assert missing_artifacts_df.empty

assert len(required_tables) == 42
assert len(required_figures) == 5
assert len(final_verification_df) == 53

print("\nConsolidated artifact verification")

display(
    final_verification_df.groupby(['Type', 'Status'])
    .size()
    .reset_index(name='Artifacts')
)


# Verify persisted numerical evidence

saved_ranking_df = pd.read_csv(
    TABLES_DIR / 'classical_validation_ranking.csv'
)

assert saved_ranking_df['Run ID'].tolist() == (
    classical_ranking_df['Run ID'].tolist()
)

np.testing.assert_allclose(
    saved_ranking_df['Macro AP'],
    classical_ranking_df['Macro AP'],
    rtol=0,
    atol=1e-12
)

saved_pr_df = pd.read_csv(
    TABLES_DIR / 'classical_selected_pr_curves.csv.gz'
)

assert len(saved_pr_df) == len(selected_pr_curves_df)

assert saved_pr_df.groupby(
    ['Run ID', 'Label']
).ngroups == 12

assert saved_pr_df['Precision'].between(0, 1).all()
assert saved_pr_df['Recall'].between(0, 1).all()

print("Saved leaderboard and PR curve evidence: Verified")

del saved_pr_df


# Prepare methodological decisions

new_decisions = [
    {
        'decision_id': 'D048',
        'component': 'Classical artifact consolidation',
        'decision': (
            'Normalize historical threshold configuration formats '
            'in memory without modifying source artifacts'
        ),
        'justification': (
            'Five experiments use thresholds and six use '
            'optimized_thresholds. All eleven experiments '
            'reproduce their saved predictions, thresholds '
            'and registered metrics with zero differences.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D049',
        'component': 'Classical algorithm comparison',
        'decision': (
            'Distinguish matched word-feature algorithm comparisons '
            'from overall configuration comparisons'
        ),
        'justification': (
            'Linear SVM leads the five common word-feature '
            'baselines, while balanced combined Logistic Regression '
            'leads the eleven evaluated configurations.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D050',
        'component': 'Classical improvement strategies',
        'decision': (
            'Retain positive and negative representation, '
            'weighting and threshold optimization findings'
        ),
        'justification': (
            'Combined features improve both linear classifiers. '
            'Balanced weighting improves Logistic Regression '
            'but reduces Linear SVM Macro AP slightly. '
            'Per-label threshold optimization improves validation '
            'Macro F1 for all eleven configurations.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D051',
        'component': 'Primary classical baseline selection',
        'decision': (
            f'Select {primary_run_id} as the primary classical '
            'reference and full-data scaling candidate'
        ),
        'justification': (
            f'Highest validation Macro AP of '
            f"{primary_row['Macro AP']:.6f} among eleven "
            'completed configurations. Selection follows the '
            'predefined validation-only policy.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D052',
        'component': 'Secondary classical baseline selection',
        'decision': (
            f'Retain {secondary_run_id} as a secondary '
            'classical reference'
        ),
        'justification': (
            f"Validation Macro AP of {secondary_row['Macro AP']:.6f} "
            'and the highest optimized Micro F1 among the '
            'evaluated classical configurations. Retain complementary '
            'precision, recall and runtime trade-offs.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D053',
        'component': 'Classical experiment scope completion',
        'decision': (
            'Complete representative-subset classical experiments '
            'and proceed to transformer fine-tuning'
        ),
        'justification': (
            'All eleven planned configurations have been '
            'trained, evaluated, registered and consolidated. '
            'Additional classical hyperparameter searches '
            'are deprioritized. Full-data scaling remains pending.'
        ),
        'status': 'Implemented'
    }
]


# Verify and update the decision log safely

current_decisions_df = pd.read_csv(
    DECISION_LOG_PATH,
    dtype=str,
    keep_default_na=False
)

assert current_decisions_df['decision_id'].is_unique

new_decisions_df = pd.DataFrame(
    new_decisions,
    columns=current_decisions_df.columns
)

new_decision_ids = new_decisions_df['decision_id'].tolist()

previous_decisions_df = current_decisions_df.loc[
    ~current_decisions_df['decision_id'].isin(new_decision_ids)
].copy()

assert len(previous_decisions_df) == 47

pd.testing.assert_frame_equal(
    previous_decisions_df.reset_index(drop=True),
    decisions_df.reset_index(drop=True),
    check_exact=True
)

existing_new_decisions = current_decisions_df.loc[
    current_decisions_df['decision_id'].isin(new_decision_ids)
]

if not existing_new_decisions.empty:

    expected_new_decisions = new_decisions_df.loc[
        new_decisions_df['decision_id'].isin(
            existing_new_decisions['decision_id']
        )
    ]

    pd.testing.assert_frame_equal(
        existing_new_decisions.sort_values(
            'decision_id'
        ).reset_index(drop=True),
        expected_new_decisions.sort_values(
            'decision_id'
        ).reset_index(drop=True),
        check_exact=True
    )

else:

    assert calculate_sha256(
        DECISION_LOG_PATH
    ) == decision_log_original_hash


updated_decisions_df = pd.concat(
    [previous_decisions_df, new_decisions_df],
    ignore_index=True
)

assert updated_decisions_df['decision_id'].is_unique
assert len(updated_decisions_df) == 53

temporary_path = DECISION_LOG_PATH.with_suffix('.tmp')

updated_decisions_df.to_csv(
    temporary_path,
    index=False
)

reloaded_decisions_df = pd.read_csv(
    temporary_path,
    dtype=str,
    keep_default_na=False
)

pd.testing.assert_frame_equal(
    updated_decisions_df,
    reloaded_decisions_df,
    check_exact=True
)

temporary_path.replace(DECISION_LOG_PATH)

assert calculate_sha256(REGISTRY_PATH) == registry_original_hash

print("\nNew methodological decisions")

display(new_decisions_df)


# Prepare the consolidated research leaderboard

summary_rows = [
    '| Rank | Configuration | Macro AP | Macro ROC-AUC | Macro F1 | Micro F1 |',
    '|---:|---|---:|---:|---:|---:|'
]

for _, row in classical_ranking_df.iterrows():

    summary_rows.append(
        f"| {int(row['Rank'])} "
        f"| {row['Run ID']} "
        f"| {row['Macro AP']:.6f} "
        f"| {row['Macro ROC-AUC']:.6f} "
        f"| {row['Optimized Macro F1']:.6f} "
        f"| {row['Optimized Micro F1']:.6f} |"
    )

leaderboard_text = '\n'.join(summary_rows)


# Save the consolidated research summary

summary = f"""# Notebook 03.06 - Consolidated Classical ML Comparison

## Objective

Compare classical multi-label classifiers under consistent
development partitions, feature budgets and evaluation policies.

## Experimental Scope

- Training comments: 59,997
- Validation comments: {len(y_val):,}
- Toxicity labels: 6
- Model families: 5
- Feature representations: 3
- Feature budget: {FEATURE_BUDGET:,}
- Random seed: {SEED}
- Completed configurations: 11

## Validation Leaderboard

{leaderboard_text}

Models are ranked by validation Macro Average Precision.
F1 scores use validation-selected per-label thresholds.

## Controlled Algorithm Comparison

Linear SVM is the strongest evaluated algorithm under
the common unweighted Word TF-IDF representation.

- Macro AP: {best_word_macro_ap:.6f}
- Optimized Macro F1:
  {common_baseline_ranking_df.iloc[0]['Optimized Macro F1']:.6f}

The comparison controls feature representation, feature budget,
development partitions and evaluation protocol. It does not
establish optimal achievable performance for each algorithm.

## Feature Representation

Character features improve overall Macro AP relative to
word features for both Logistic Regression and Linear SVM.

Combined Word and Character TF-IDF produces the strongest
evaluated unweighted configuration within both linear families.

The total feature budget remains fixed at 50,000.

## Class Weighting

Balanced class weighting improves Logistic Regression Macro AP
from 0.647109 to 0.671787 and optimized Macro F1 from
0.633514 to 0.659754.

For Linear SVM, balanced weighting reduces Macro AP from
0.662505 to 0.659581 and optimized Macro F1 from
0.656899 to 0.650759.

Class weighting is therefore model-dependent under the
evaluated configurations. Weighting experiments were conducted
only with combined features.

## Threshold Optimization

- Configurations with improved validation Macro F1: 11/11
- Individual label F1 improvements: 65/66
- Zero-positive prediction cases recovered: 5/5

Random Forest exhibits the largest Macro F1 improvement,
from 0.051017 to 0.586044.

Threshold selection does not modify ranking scores.
Validation-selected threshold performance requires independent
test evaluation before generalization claims.

## Nonlinear Models

Random Forest improves Macro AP from 0.421514 to 0.586785
relative to Decision Tree.

The overall training-validation AP gap decreases by 39.3631%.
Threat remains a limitation, with an increased AP gap
and slightly lower optimized F1.

Probability averaging increases the number of distinct scores,
but does not establish probability calibration.

## Primary Classical Baseline

- Run: {primary_run_id}
- Model: Logistic Regression
- Representation: Combined Word and Character TF-IDF
- Class weighting: Balanced
- Macro AP: {primary_row['Macro AP']:.6f}
- Macro ROC-AUC: {primary_row['Macro ROC-AUC']:.6f}
- Optimized Macro F1: {primary_row['Optimized Macro F1']:.6f}
- Optimized Micro F1: {primary_row['Optimized Micro F1']:.6f}

Selected using the predefined primary metric, validation Macro AP.
This configuration is the candidate for full-data scaling.

## Secondary Classical Baseline

- Run: {secondary_run_id}
- Model: Linear SVM
- Representation: Combined Word and Character TF-IDF
- Class weighting: None
- Macro AP: {secondary_row['Macro AP']:.6f}
- Macro ROC-AUC: {secondary_row['Macro ROC-AUC']:.6f}
- Optimized Macro F1: {secondary_row['Optimized Macro F1']:.6f}
- Optimized Micro F1: {secondary_row['Optimized Micro F1']:.6f}

This model provides a complementary reference with higher
optimized Micro F1 and lower recorded training time.

## Computational Analysis

Six configurations are Pareto-efficient with respect to
validation Macro AP and recorded training time.

Runtime measurements are descriptive and exclude feature
extraction. Parallelization differs across algorithms.
Prediction times use precomputed features rather than raw text.
Model sizes exclude vectorizers.

## Reproducibility

- Reconstructed experiments: 11/11
- Maximum threshold reconstruction difference: 0.0
- Maximum Macro AP reconstruction error: 0.0
- Maximum Macro F1 reconstruction error: 0.0
- Original source artifact fingerprints verified: 44
- Historical standard manifests verified: 10/11
- Selected vectorizer dependencies verified: 2
- Consolidated analysis tables: {len(required_tables)}
- Research figures: {len(required_figures)}
- Configurations: {len(required_configs)}
- Logs: {len(required_logs)}
- Required output artifacts: {len(final_verification_df)}
- Missing artifacts: 0

Naive Bayes has no historical manifest in the expected format.
Its current critical artifacts are fingerprinted, but their
historical integrity cannot be established retrospectively
from these new fingerprints alone.

Historical threshold JSON formats are normalized only
in memory. Original threshold files remain unchanged.

## Limitations

- Single validation split and random seed.
- No exhaustive classical hyperparameter optimization.
- No statistical significance established.
- Validation threshold performance may be optimistic.
- SVM decision margins are not calibrated probabilities.
- Pooled micro ranking metrics require caution across score types.
- Runtime measurements have not been repeated.
- Subset-based training does not establish full-data performance.

## Research Status

All eleven planned representative-subset classical experiments
are completed and registered.

No additional classical training was performed in this notebook.

The official test partition was not used in these experiments
for training, threshold optimization or model selection.

The final project model has not been frozen.

Full-data scaling of the strongest classical configuration
remains pending and requires refitting the vectorizers,
retraining the classifier and selecting new validation thresholds.

## Next Stage

Notebook 04 - DistilBERT.

Fine-tune and evaluate the transformer using controlled loss
configurations, consistent development partitions and
validation-based model selection.
"""

summary_path = RESULTS_DIR / 'summary.md'

summary_path.write_text(
    summary,
    encoding='utf-8'
)

assert summary_path.read_text(
    encoding='utf-8'
) == summary


# Include the research summary in the final inventory

summary_record = pd.DataFrame([{
    'Type': 'Research Summary',
    'Filename': summary_path.name,
    'Path': str(summary_path.relative_to(PROJECT_ROOT)),
    'Size Bytes': summary_path.stat().st_size,
    'SHA256': calculate_sha256(summary_path),
    'Status': 'Available'
}])

final_verification_df = pd.concat(
    [final_verification_df, summary_record],
    ignore_index=True
)

assert len(final_verification_df) == 54

final_verification_df.to_csv(
    TABLES_DIR / 'classical_final_artifact_verification.csv',
    index=False
)


# Perform final consistency checks

final_registry_df = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

final_decisions_df = pd.read_csv(
    DECISION_LOG_PATH,
    dtype=str,
    keep_default_na=False
)

assert len(final_registry_df) == 11
assert final_registry_df['run_id'].is_unique
assert final_registry_df['status'].eq('Completed').all()

assert len(final_decisions_df) == 53
assert final_decisions_df['decision_id'].is_unique

for _, artifact in final_verification_df.iterrows():

    path = PROJECT_ROOT / artifact['Path']

    assert path.is_file()
    assert calculate_sha256(path) == artifact['SHA256']

assert calculate_sha256(REGISTRY_PATH) == registry_original_hash

assert not saved_selection['official_test_used']
assert not saved_selection['final_project_model_frozen']


# Summarize the completed classical ML stage

print("\nClassical ML completion summary")

print("Model families:", final_registry_df['model'].nunique())
print("Completed classical configurations:", len(final_registry_df))
print("Independently reconstructed experiments:", len(classical_results))
print("Original source artifact fingerprints verified:", len(source_artifact_df))

print("Consolidated analysis tables:", len(required_tables))
print("Research figures:", len(required_figures))
print("Configuration files:", len(required_configs))
print("Reconstruction logs:", len(required_logs))
print("Research summaries: 1")

print("Total verified consolidated artifacts:", len(final_verification_df))
print("Missing artifacts: 0")

print("Previous methodological decisions:", len(previous_decisions_df))
print("New methodological decisions:", len(new_decisions_df))
print("Total methodological decisions:", len(final_decisions_df))

print("Duplicate experiment IDs:", final_registry_df['run_id'].duplicated().sum())

print("\nPrimary classical baseline:", primary_run_id)
print("Validation Macro AP:", round(primary_row['Macro AP'], 6))

print("Secondary classical baseline:", secondary_run_id)
print("Validation Macro AP:", round(secondary_row['Macro AP'], 6))

print("\nOriginal experiment registry: Unchanged")
print("Original model and threshold artifacts: Preserved")
print("Classical selection configuration: Verified")
print("Research summary: Saved")
print("Full-data scaling: Pending")
print("Additional model training: No")
print("Official test used: No")
print("Final project model frozen: No")

gc.collect()

print("\nClassical ML comparison notebook completed successfully.")
print("Next notebook: 04_distilbert")